# 06 Regressions

Equations 8 and 12-15. Inputs: `data/processed/analysis_panel_2018_2025.parquet`, `data/features/nlp_features.parquet`, `data/features/novelty_placebo.csv`. Produces Tables 3, 6, 7, 8 and 10-14 and Appendix Tables VII.1 and VIII.1-VIII.4 (`output/tables/`).

## 1 Setup

In [27]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from linearmodels import PanelOLS

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_rows', 120)

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT   = Path.cwd().parent
OUTPUT = ROOT / 'output' / 'tables'
OUTPUT.mkdir(parents=True, exist_ok=True)

# ── Variable lists (SPEC 1) ────────────────────────────────────────────────
NLP_VARS = ['intensity', 'geo_tone', 'specificity', 'forward_look',
            'uncertainty', 'novelty_geo']
CONTENT_VARS = ['specificity', 'forward_look', 'uncertainty',
                'geo_tone', 'novelty_geo']
ROBUSTNESS_NOVELTY = 'novelty_full_item1a'

BASELINE_CONTROLS = ['L_log_at', 'L_leverage_w', 'L_roa_w',
                     'L_tobin_q_w', 'L_cash_flow_ratio_w',
                     'L_sales_growth_w', 'L_log_analyst_cov',
                     'L_firm_risk_w']
TEXT_LENGTH_CTRL = 'L_log_item1a_tokens'  # constructed in Cell 5
ALL_CONTROLS = BASELINE_CONTROLS + [TEXT_LENGTH_CTRL]

PRED_OUTCOMES = ['firm_risk_w', 'beta_w', 'ret_ann_w', 'leverage_w',
                 'cash_flow_ratio_w', 'operating_cf_ratio_w']

# ── Significance stars (*** <0.01, ** <0.05, * <0.10) ───────
def stars(p: float) -> str:
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.10: return '*'
    return ''

# ── Core regression helpers ────────────────────────────────────────────────

def run_panel_fe(df, dv, rhs_vars, entity_col='cik', time_col='fiscal_year'):
    """PanelOLS with firm + year FE and firm-clustered SEs.
    Returns (result, n) or (None, n) if sample too small.
    """
    all_cols = list(dict.fromkeys([entity_col, time_col, dv] + rhs_vars))
    sub = df[[c for c in all_cols if c in df.columns]].dropna()
    if len(sub) < 20:
        return None, len(sub)
    sub = sub.set_index([entity_col, time_col])
    y = sub[dv]
    X = sub[[c for c in rhs_vars if c in sub.columns]]
    mod = PanelOLS(y, X, entity_effects=True, time_effects=True, drop_absorbed=True)
    res = mod.fit(cov_type='clustered', cluster_entity=True)
    return res, len(sub)


def fmt_res(res, n, dv='', label=''):
    """Extract coefficient table from a linearmodels PanelOLS result."""
    rows = []
    for var in res.params.index:
        p = float(res.pvalues[var])
        rows.append({
            'label': label,
            'dv': dv,
            'variable': var,
            'coef': float(res.params[var]),
            'se': float(res.std_errors[var]),
            'tstat': float(res.tstats[var]),
            'pval': p,
            'stars': stars(p),
            'n': n,
            'within_r2': float(res.rsquared),
        })
    return pd.DataFrame(rows)


def run_interaction_reg(df, dv, nlp_var, mod_var, add_controls,
                        entity_col='cik', time_col='fiscal_year'):
    """SPEC 5: Within-estimation-sample standardise interactants, form interaction,
    run PanelOLS with firm + year FE and firm-clustered SEs.
    Returns (result, n, std_check_df) or (None, 0, None) if insufficient data.
    std_check_df has rows [mean, std] for the two standardised interactants.
    """
    cols = list(dict.fromkeys(
        [entity_col, time_col, dv, nlp_var, mod_var] + add_controls))
    sub = df[[c for c in cols if c in df.columns]].dropna()
    if len(sub) < 20:
        return None, 0, None

    sub = sub.copy()
    nlp_z  = f'{nlp_var}_std'
    mod_z  = f'{mod_var}_std'
    int_z  = f'{nlp_var}_x_{mod_var}'

    # Step 2 (SPEC 5): z-scores within estimation sample
    sub[nlp_z] = (sub[nlp_var] - sub[nlp_var].mean()) / sub[nlp_var].std()
    sub[mod_z] = (sub[mod_var] - sub[mod_var].mean()) / sub[mod_var].std()
    # Step 3: interaction as product of z-scores
    sub[int_z] = sub[nlp_z] * sub[mod_z]

    std_check = sub[[nlp_z, mod_z]].agg(['mean', 'std'])

    rhs = [c for c in [nlp_z, mod_z, int_z] + add_controls if c in sub.columns]
    sub2 = sub.set_index([entity_col, time_col])
    y = sub2[dv]
    X = sub2[rhs]
    mod_obj = PanelOLS(y, X, entity_effects=True, time_effects=True, drop_absorbed=True)
    res = mod_obj.fit(cov_type='clustered', cluster_entity=True)
    return res, len(sub), std_check


print('Setup complete.')
print(f'NLP_VARS:      {NLP_VARS}')
print(f'CONTENT_VARS:  {CONTENT_VARS}')
print(f'PRED_OUTCOMES: {PRED_OUTCOMES}')
print(f'ALL_CONTROLS:  {ALL_CONTROLS}')
print(f'Output dir:    {OUTPUT}')

Setup complete.
NLP_VARS:      ['intensity', 'geo_tone', 'specificity', 'forward_look', 'uncertainty', 'novelty_geo']
CONTENT_VARS:  ['specificity', 'forward_look', 'uncertainty', 'geo_tone', 'novelty_geo']
PRED_OUTCOMES: ['firm_risk_w', 'beta_w', 'ret_ann_w', 'leverage_w', 'cash_flow_ratio_w', 'operating_cf_ratio_w']
ALL_CONTROLS:  ['L_log_at', 'L_leverage_w', 'L_roa_w', 'L_tobin_q_w', 'L_cash_flow_ratio_w', 'L_sales_growth_w', 'L_log_analyst_cov', 'L_firm_risk_w', 'L_log_item1a_tokens']
Output dir:    c:\Users\Sören Schlißke\Dateien\Uni\MACFin\Thesis\Claude\output\tables


## 2 Data loading and preparation

In [28]:
# ── Load raw files ─────────────────────────────────────────────────────────
ap  = pd.read_parquet(ROOT / 'data/processed/analysis_panel_2018_2025.parquet')
nlp = pd.read_parquet(ROOT / 'data/features/nlp_features.parquet')

print(f'analysis_panel shape:  {ap.shape}')
print(f'nlp_features shape:    {nlp.shape}')

# ── Merge: nlp_features is authoritative for all NLP columns ──────────────
# Drop any NLP cols already in analysis_panel before merging to prevent conflicts
NLP_OVERRIDE = ['intensity', 'geo_tone', 'specificity', 'forward_look',
                'uncertainty', 'novelty_geo', 'novelty_full_item1a',
                'boilerplate', 'boilerplate_geo', 'token_count',
                'geo_sentence_count', 'geo_token_count']

ap_clean   = ap.drop(columns=[c for c in NLP_OVERRIDE if c in ap.columns],
                     errors='ignore')
nlp_subset = nlp[['cik', 'fiscal_year'] +
                 [c for c in NLP_OVERRIDE if c in nlp.columns]]

panel = ap_clean.merge(nlp_subset, on=['cik', 'fiscal_year'], how='left')

print(f'\nMerged panel shape: {panel.shape}')
print(f'Unique firms: {panel["cik"].nunique():,}')
print(f'Fiscal years: {sorted(panel["fiscal_year"].unique())}')

# ── Verify expected NLP columns ────────────────────────────────────────────
EXPECTED_NLP = ['intensity', 'geo_tone', 'specificity', 'forward_look',
                'uncertainty', 'novelty_geo', 'novelty_full_item1a',
                'boilerplate', 'boilerplate_geo']

print('\nNLP column presence and null counts:')
for col in EXPECTED_NLP:
    if col in panel.columns:
        null_n = panel[col].isna().sum()
        pct    = null_n / len(panel) * 100
        print(f'  {col:<35s} {null_n:5d} nulls  ({pct:.1f}%)')
    else:
        print(f'  {col:<35s} MISSING')

print('\nBaseline controls null counts:')
for col in BASELINE_CONTROLS:
    null_n = panel[col].isna().sum()
    print(f'  {col:<35s} {null_n:5d} nulls')

analysis_panel shape:  (10962, 169)
nlp_features shape:    (10962, 40)

Merged panel shape: (10962, 169)
Unique firms: 1,488
Fiscal years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

NLP column presence and null counts:
  intensity                               0 nulls  (0.0%)
  geo_tone                             1473 nulls  (13.4%)
  specificity                          1473 nulls  (13.4%)
  forward_look                         1473 nulls  (13.4%)
  uncertainty                          1473 nulls  (13.4%)
  novelty_geo                          3102 nulls  (28.3%)
  novelty_full_item1a                  1560 nulls  (14.2%)
  boilerplate                          1560 nulls  (14.2%)
  boilerplate_geo                      3102 nulls  (28.3%)

Baseline controls null counts:
  L_log_at                              139 nulls
  L_leverage_w                          139 nulls
  L_roa_w                       

In [29]:
ap.describe()

,cik,fiscal_year,token_count,sentence_count,geo_sentence_count,geo_token_count,geo_sentence_share,item1a_tokens,log_item1a_tokens,item1a_sentences,geo_tokens,geo_sentences,intensity,geo_tone,n_negative_hits_geo,n_mitigation_hits_geo,specificity,n_geo_entities,n_unique_geo_entities,n_channel_hits,forward_look,n_forward_geo_sentences,n_mitigation_geo_sentences,uncertainty,n_uncertainty_hits_geo,ner_geo_entities_trf,n_geo_entities_sm,boilerplate,novelty_full_item1a,boilerplate_geo,novelty_geo,fyear_cs,sich,naics,ff48_industry,fyr,datadate,permno,mktcap_usd_bn,log_mktcap,ret_ann,beta,ret_ann_w,beta_w,at,sale,revt,ni,ib,oibdp,oiadp,dltt,dlc,lt,ceq,seq,act,lct,che,ppent,capx,xrd,dp,csho,prcc_f,mkvalt,dvt,pifo,pi,log_at,log_sale,leverage,leverage_w,net_leverage,net_leverage_w,debt_to_equity,debt_to_equity_w,roa,roa_w,roe,roe_w,ebitda_margin,ebitda_margin_w,profit_margin,profit_margin_w,asset_turnover,cash_ratio,cash_ratio_w,current_ratio,current_ratio_w,capex_intensity,capex_intensity_w,rd_intensity,rd_intensity_w,rd_missing,tangibility,tangibility_w,depreciation_rate,tobin_q,tobin_q_w,book_to_market,book_to_market_w,foreign_income_share,foreign_income_share_w,dividend_yield,dividend_yield_w,payout_ratio,payout_ratio_w,sales_growth,sales_growth_w,firm_risk,firm_risk_w,cash_flow_ratio,cash_flow_ratio_w,operating_cf_ratio,operating_cf_ratio_w,log_n_biz_seg,foreign_rev_share,foreign_rev_share_w,n_geo_segments,seg_total_rev,seg_foreign_rev,has_geo_seg,has_inst_own,analyst_coverage,log_analyst_cov,eps_median,eps_dispersion,eps_uncertainty,has_ibes,vix_annual,rev_share,hhi,high_litigation,L_log_at,L_leverage_w,L_roa_w,L_tobin_q_w,L_cash_ratio_w,L_rd_intensity_w,L_tangibility_w,L_foreign_rev_share_w,L_log_analyst_cov,L_ret_ann_w,L_beta_w,L_log_mktcap,L_book_to_market_w,L_sales_growth_w,L_firm_risk_w,L_cash_flow_ratio_w,L_operating_cf_ratio_w
count,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,9489.000000,10962.000000,10962.000000,9489.000000,10962.000000,10962.000000,10962.000000,9489.000000,10962.000000,10962.000000,9489.000000,10962.000000,10962.000000,0.000000,9402.000000,9402.000000,7860.000000,7860.000000,10961.000000,10542.000000,10493.000000,10921.000000,10921.000000,10921,10870.000000,10869.000000,10869.000000,10813.000000,10724.000000,10813.000000,10724.000000,10916.000000,10915.000000,10915.000000,10915.000000,10915.000000,10234.000000,10915.000000,10884.000000,10909.000000,10889.000000,10916.000000,10916.000000,8467.000000,8467.000000,10915.000000,10300.000000,10864.000000,5987.000000,10140.000000,10906.000000,10891.000000,10197.000000,10823.000000,6860.000000,10915.000000,10916.000000,10915.000000,10916.000000,10916.000000,10916.000000,10916.000000,10916.000000,10916.000000,10915.000000,10915.000000,10915.000000,10915.000000,10227.000000,10227.000000,10908.000000,10908.000000,10915.000000,10915.000000,10915.000000,8467.000000,8467.000000,10916.000000,10916.000000,10916.000000,10916.000000,10921.000000,10300.000000,10300.000000,10300.000000,10886.000000,10886.000000,10886.000000,10886.000000,6859.000000,6859.000000,10882.000000,10882.000000,10915.000000,10915.000000,10776.000000,10776.000000,9580.000000,9580.000000,10916.000000,10916.000000,10916.000000,10916.000000,10921.000000,6859.000000,6859.000000,0.000000,0.000000,0.000000,10921.000000,10921.000000,10743.000000,10921.000000,10743.000000,10453.000000,10453.000000,10921.000000,10921.000000,10915.000000,10921.000000,10921.000000,10823.000000,10823.000000,10820.000000,10668.000000,10823.000000,10823.000000,10204.000000,6703.000000,10844.000000,10582.000000,10476.000000,10645.000000,10667.000000,9596.000000,8235.000000,10823.000000,10823.000000
mean,964111.227422,2021.650246,23751.346926,633.003558,12.615034,958.198321,0.023160,23751.346926,9.877887,633.003558,958.198321,12.615034,0.047263,0.033294,34.553731,4.466429,0.008594,8.304598,2.855592,0.916621,0.058

In [30]:
nlp.describe()

,cik,fiscal_year,token_count,sentence_count,geo_sentence_count,geo_token_count,geo_sentence_share,item1a_tokens,log_item1a_tokens,item1a_sentences,geo_tokens,geo_sentences,intensity,geo_tone,n_negative_hits_geo,n_mitigation_hits_geo,specificity,n_geo_entities,n_unique_geo_entities,n_channel_hits,forward_look,n_forward_geo_sentences,n_mitigation_geo_sentences,uncertainty,n_uncertainty_hits_geo,ner_geo_entities_trf,n_geo_entities_sm,boilerplate,novelty_full_item1a,boilerplate_geo,novelty_geo
count,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,10962.000000,9489.000000,10962.000000,10962.000000,9489.000000,10962.000000,10962.000000,10962.000000,9489.000000,10962.000000,10962.000000,9489.000000,10962.000000,10962.000000,0.000000,9402.000000,9402.000000,7860.000000,7860.000000
mean,964111.227422,2021.650246,23751.346926,633.003558,12.615034,958.198321,0.023160,23751.346926,9.877887,633.003558,958.198321,12.615034,0.047263,0.033294,34.553731,4.466429,0.008594,8.304598,2.855592,0.916621,0.058556,0.760993,2.234720,1.121609,13.694764,8.304598,NaN,0.886068,0.113932,0.831959,0.168041
std,551145.430323,2.263724,13507.087761,343.888841,12.534406,966.631661,0.020591,13507.087761,0.679677,343.888841,966.631661,12.534406,0.041700,0.016751,36.845460,5.950532,0.010590,16.763672,3.861086,3.395766,0.091712,1.401506,2.836979,0.422902,13.702013,16.763672,NaN,0.100801,0.100801,0.150468,0.150468
min,1750.000000,2018.000000,279.000000,13.000000,0.000000,0.000000,0.000000,279.000000,5.634790,13.000000,0.000000,0.000000,0.000000,-0.065574,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.104874,0.000000,0.022887,0.000000
25%,731012.000000,2020.000000,11597.000000,336.000000,5.000000,308.000000,0.008373,11597.000000,9.358588,336.000000,308.000000,5.000000,0.015676,0.022124,11.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.846154,5.000000,0.000000,NaN,0.865154,0.053170,0.765561,0.059108
50%,1022408.000000,2022.000000,22690.000000,606.000000,9.000000,676.000000,0.017333,22690.000000,10.029724,606.000000,676.000000,9.000000,0.036228,0.031761,24.000000,2.000000,0.006122,3.000000,2.000000,0.000000,0.000000,0.000000,1.000000,1.083333,10.000000,3.000000,NaN,0.910420,0.089580,0.874618,0.125382
75%,1418121.000000,2024.000000,33995.750000,867.000000,16.000000,1283.000000,0.031797,33995.750000,10.434020,867.000000,1283.000000,16.000000,0.067721,0.042759,46.000000,6.000000,0.013289,10.000000,4.000000,0.000000,0.100000,1.000000,3.000000,1.333333,18.000000,10.000000,NaN,0.946830,0.134846,0.940892,0.234439
max,2082866.000000,2025.000000,171698.000000,4357.000000,155.000000,10637.000000,0.212329,171698.000000,12.053498,4357.000000,10637.000000,155.000000,0.332147,0.121795,427.000000,82.000000,0.266667,369.000000,68.000000,86.000000,1.000000,18.000000,38.000000,5.000000,169.000000,369.000000,NaN,1.000000,0.895126,1.000000,0.977113


In [31]:
# Sort once; all subsequent lag/lead operations use this ordering
panel = panel.sort_values(['cik', 'fiscal_year']).reset_index(drop=True)

# ── SPEC 2: L_log_item1a_tokens ───────────────────────────────────────────
panel['log_item1a_tokens']   = np.log1p(panel['token_count'])

prev_year = panel.groupby('cik')['fiscal_year'].shift(1)
panel['L_log_item1a_tokens'] = (panel.groupby('cik')['log_item1a_tokens'].shift(1)
                                .where(prev_year == panel['fiscal_year'] - 1))


llit = panel['L_log_item1a_tokens']
print(f'L_log_item1a_tokens: nulls={llit.isna().sum():,}  '
      f'mean={llit.mean():.3f}  std={llit.std():.3f}')

for c in NLP_VARS + [ROBUSTNESS_NOVELTY]:
    lo, hi = panel[c].quantile([0.01, 0.99])
    panel[c] = panel[c].clip(lo, hi)

# Robustness flag: original thin-denominator thresholds (5 sentences / 200 tokens).
# Main specification uses 3 / 100 (already applied in 05).
panel['thin_strict'] = (panel['geo_sentence_count'] < 5) | (panel['geo_token_count'] < 200)
print(f"thin_strict (5/200): {panel['thin_strict'].sum():,} firm-years")
print('rd_missing in panel:', 'rd_missing' in panel.columns,
      '| high_litigation in panel:', 'high_litigation' in panel.columns)


# ── SPEC 3: PropCost composite (full-panel standardisation as approximation) 
# Within each interaction regression, standardisation is redone on the
# estimation subsample (SPEC 5 / run_interaction_reg).
panel['rd_z']       = ((panel['rd_intensity_w'] - panel['rd_intensity_w'].mean())
                       / panel['rd_intensity_w'].std())
panel['ebitda_z_neg'] = -1.0 * (
    (panel['ebitda_margin_w'] - panel['ebitda_margin_w'].mean())
    / panel['ebitda_margin_w'].std()
)
panel['prop_cost']     = (panel['rd_z'] + panel['ebitda_z_neg']) / 2.0
panel['prop_cost_std'] = (
    (panel['prop_cost'] - panel['prop_cost'].mean()) / panel['prop_cost'].std()
)
print(f'prop_cost: nulls={panel["prop_cost"].isna().sum():,}  '
      f'prop_cost_std mean={panel["prop_cost_std"].mean():.4f}  '
      f'std={panel["prop_cost_std"].std():.4f}')

# ── Consecutive-year flag (used in persistence analysis) ───────────────────
panel['L_fiscal_year'] = panel.groupby('cik')['fiscal_year'].shift(1)
panel['is_consec']     = (panel['fiscal_year'] - panel['L_fiscal_year'] == 1)


# ── CHECKPOINT 1 ──────────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('CHECKPOINT 1')
print('=' * 65)
print(f'Panel size: {len(panel):,} firm-year observations')
print(f'Unique firms: {panel["cik"].nunique():,}')
print(f'Fiscal years: {sorted(panel["fiscal_year"].unique())}')
print()
print('NLP columns:')
for col in EXPECTED_NLP:
    if col in panel.columns:
        null_n = panel[col].isna().sum()
        tag    = 'OK'
        print(f'  {col:<35s} {tag}  {null_n} nulls')
    else:
        print(f'  {col:<35s} MISSING')
print()
print(f'L_log_item1a_tokens constructed: {"L_log_item1a_tokens" in panel.columns}')
print(f'prop_cost composite constructed: {"prop_cost" in panel.columns}')
print()
print('Missing-value counts per NLP variable:')
for col in NLP_VARS + [ROBUSTNESS_NOVELTY]:
    null_n = panel[col].isna().sum() if col in panel.columns else -1
    print(f'  {col:<25s} {null_n}')
print('Missing-value counts per baseline control:')
for col in BASELINE_CONTROLS:
    null_n = panel[col].isna().sum()
    print(f'  {col:<25s} {null_n}')



L_log_item1a_tokens: nulls=1,560  mean=9.874  std=0.687
thin_strict (5/200): 2,882 firm-years
rd_missing in panel: True | high_litigation in panel: True
prop_cost: nulls=735  prop_cost_std mean=0.0000  std=1.0000

CHECKPOINT 1
Panel size: 10,962 firm-year observations
Unique firms: 1,488
Fiscal years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

NLP columns:
  intensity                           OK  0 nulls
  geo_tone                            OK  1473 nulls
  specificity                         OK  1473 nulls
  forward_look                        OK  1473 nulls
  uncertainty                         OK  1473 nulls
  novelty_geo                         OK  3102 nulls
  novelty_full_item1a                 OK  1560 nulls
  boilerplate                         OK  1560 nulls
  boilerplate_geo                     OK  3102 nulls

L_log_item1a_tokens constructed: True
prop_cost composite constructed: True

Mi

In [32]:
panel[
    ['intensity', 'geo_tone', 'specificity', 'forward_look',
     'uncertainty', 'novelty_geo']
].mean().round(3)

intensity      0.047000
geo_tone       0.033000
specificity    0.008000
forward_look   0.057000
uncertainty    1.119000
novelty_geo    0.167000
dtype: float64

## 3 Determinants and composition of disclosure

### 3A Intensity model (Equation 8, Table 8)

In [33]:
res_3a, n_3a = run_panel_fe(panel, dv='intensity', rhs_vars=ALL_CONTROLS)
df_3a = fmt_res(res_3a, n_3a, dv='intensity', label='3A_intensive')

print('Section 3A: Intensive-Margin Model (Eq. 8)')
print(f'N = {n_3a:,}  |  Within R² = {res_3a.rsquared:.4f}')
print()
display_cols = ['variable', 'coef', 'se', 'tstat', 'pval', 'stars']
print(df_3a[display_cols].to_string(index=False))

Section 3A: Intensive-Margin Model (Eq. 8)
N = 8,039  |  Within R² = 0.0088

           variable      coef       se     tstat     pval stars
           L_log_at  0.000038 0.001618  0.023699 0.981094      
       L_leverage_w -0.002014 0.004931 -0.408467 0.682944      
            L_roa_w  0.027469 0.036062  0.761730 0.446249      
        L_tobin_q_w  0.000004 0.000418  0.009741 0.992228      
L_cash_flow_ratio_w -0.033301 0.036933 -0.901681 0.367260      
   L_sales_growth_w  0.001286 0.001131  1.136704 0.255704      
  L_log_analyst_cov  0.001521 0.001343  1.132107 0.257631      
      L_firm_risk_w -0.000750 0.013082 -0.057355 0.954264      
L_log_item1a_tokens -0.004797 0.001059 -4.530714 0.000006   ***


### 3B Content dimensions on intensity (Equation 12)

In [34]:
results_3b = {}
n_3b       = {}
dfs_3b     = []

for cv in CONTENT_VARS:
    rhs = ['intensity'] + ALL_CONTROLS
    res, n = run_panel_fe(panel, dv=cv, rhs_vars=rhs)
    results_3b[cv] = res
    n_3b[cv]       = n
    if res is not None:
        dfs_3b.append(fmt_res(res, n, dv=cv, label=f'3B_{cv}'))

df_3b = pd.concat(dfs_3b, ignore_index=True)

print('Section 3B: Conditional Content Models (Eq. 9)')
print()
for cv in CONTENT_VARS:
    res = results_3b[cv]
    r2  = f'{res.rsquared:.4f}' if res is not None else 'N/A'
    print(f'  DV = {cv:<20s}  N = {n_3b[cv]:,}  Within R² = {r2}')
    if res is not None and 'intensity' in res.params.index:
        coef = float(res.params['intensity'])
        p    = float(res.pvalues['intensity'])
        se   = float(res.std_errors['intensity'])
        print(f'    intensity: coef={coef:+.4f}  se={se:.4f}  p={p:.3f}  {stars(p)}')

print()
print('Full coefficient table stored in df_3b.')

Section 3B: Conditional Content Models (Eq. 9)

  DV = specificity           N = 7,107  Within R² = 0.0041
    intensity: coef=+0.0067  se=0.0065  p=0.296  
  DV = forward_look          N = 7,107  Within R² = 0.0074
    intensity: coef=+0.2249  se=0.0614  p=0.000  ***
  DV = uncertainty           N = 7,107  Within R² = 0.0188
    intensity: coef=+1.3589  se=0.2736  p=0.000  ***
  DV = geo_tone              N = 7,107  Within R² = 0.0106
    intensity: coef=-0.0340  se=0.0117  p=0.004  ***
  DV = novelty_geo           N = 6,817  Within R² = 0.0195
    intensity: coef=+0.7761  se=0.1416  p=0.000  ***

Full coefficient table stored in df_3b.


In [35]:
df_3b.to_csv('3B_results.csv')

### 3C Interactions with proprietary costs and litigation risk (Equation 13)

In [36]:
results_3c = {}
n_3c       = {}
dfs_3c     = []

for cv in CONTENT_VARS:
    res, n, std_check = run_interaction_reg(
        panel, dv=cv,
        nlp_var='intensity', mod_var='prop_cost',
        add_controls=ALL_CONTROLS + ['rd_missing']
    )
    results_3c[cv] = res
    n_3c[cv]       = n
    if res is not None:
        dfs_3c.append(fmt_res(res, n, dv=cv, label=f'3C_{cv}'))

df_3c = pd.concat(dfs_3c, ignore_index=True)

# ── CHECKPOINT 2 ──────────────────────────────────────────────────────────
print('=' * 65)
print('CHECKPOINT 2')
print('=' * 65)
print(f'Section 3A  N = {n_3a:,}')
print()
print('Section 3B  N per DV:')
for cv in CONTENT_VARS:
    print(f'  {cv:<20s}  N = {n_3b[cv]:,}')
print()
print('Section 3C  H2 interaction (intensity x prop_cost):')
for cv in CONTENT_VARS:
    res = results_3c[cv]
    n   = n_3c[cv]
    int_key = [k for k in res.params.index if 'x_prop_cost' in str(k)] if res else []
    if int_key:
        coef = float(res.params[int_key[0]])
        p    = float(res.pvalues[int_key[0]])
        sig  = stars(p)
        print(f'  {cv:<20s}  N = {n:,}  coef = {coef:+.5f}  p = {p:.3f}  {sig}')
    else:
        print(f'  {cv:<20s}  N = {n:,}  interaction term missing')

CHECKPOINT 2
Section 3A  N = 8,039

Section 3B  N per DV:
  specificity           N = 7,107
  forward_look          N = 7,107
  uncertainty           N = 7,107
  geo_tone              N = 7,107
  novelty_geo           N = 6,817

Section 3C  H2 interaction (intensity x prop_cost):
  specificity           N = 6,671  coef = -0.00005  p = 0.790  
  forward_look          N = 6,671  coef = -0.00178  p = 0.382  
  uncertainty           N = 6,671  coef = -0.00208  p = 0.773  
  geo_tone              N = 6,671  coef = +0.00026  p = 0.408  
  novelty_geo           N = 6,412  coef = -0.00100  p = 0.800  


In [37]:
df_3c.to_csv('3C_outputs.csv')

In [38]:
# H2 robustness: litigation-risk moderator instead of PropCost
results_3c_lit, dfs_3c_lit = {}, []
for cv in CONTENT_VARS:
    res, n, _ = run_interaction_reg(panel, dv=cv, nlp_var='intensity',
                                    mod_var='high_litigation',
                                    add_controls=ALL_CONTROLS)
    results_3c_lit[cv] = res
    if res is not None:
        k = [x for x in res.params.index if 'x_high_litigation' in str(x)][0]
        print(f'{cv:<15s} N={n:,}  interaction={res.params[k]:+.5f}  p={res.pvalues[k]:.3f}')
        dfs_3c_lit.append(fmt_res(res, n, dv=cv, label=f'3C_lit_{cv}'))
pd.concat(dfs_3c_lit).to_csv('3C_litigation.csv')

specificity     N=7,075  interaction=+0.00025  p=0.155
forward_look    N=7,075  interaction=-0.00047  p=0.789
uncertainty     N=7,075  interaction=-0.00861  p=0.315
geo_tone        N=7,075  interaction=-0.00015  p=0.683
novelty_geo     N=6,787  interaction=-0.00184  p=0.651


## 4 Predictive regressions (Equation 15)

In [39]:
# Construct t+1 lead outcomes via groupby.shift(-1) (SPEC 6)
next_year = panel.groupby('cik')['fiscal_year'].shift(-1)
for outcome in PRED_OUTCOMES:
    panel[f'lead_{outcome}'] = (panel.groupby('cik')[outcome].shift(-1)
                                .where(next_year == panel['fiscal_year'] + 1))

print('Lead outcomes constructed (t+1 values):')
for outcome in PRED_OUTCOMES:
    lead_col = f'lead_{outcome}'
    non_null = panel[lead_col].notna().sum()
    null_n   = panel[lead_col].isna().sum()
    print(f'  {lead_col:<30s}: {non_null:,} non-null  ({null_n:,} null — last year per firm drops)')

Lead outcomes constructed (t+1 values):
  lead_firm_risk_w              : 9,233 non-null  (1,729 null — last year per firm drops)
  lead_beta_w                   : 9,318 non-null  (1,644 null — last year per firm drops)
  lead_ret_ann_w                : 9,330 non-null  (1,632 null — last year per firm drops)
  lead_leverage_w               : 9,364 non-null  (1,598 null — last year per firm drops)
  lead_cash_flow_ratio_w        : 9,364 non-null  (1,598 null — last year per firm drops)
  lead_operating_cf_ratio_w     : 9,364 non-null  (1,598 null — last year per firm drops)


### 4A Intensity only

In [40]:
results_4a = {}
n_4a       = {}
dfs_4a     = []

for outcome in PRED_OUTCOMES:
    lead_col = f'lead_{outcome}'
    rhs = ['intensity'] + ALL_CONTROLS + [outcome]  # contemporaneous outcome as lagged DV
    res, n = run_panel_fe(panel, dv=lead_col, rhs_vars=rhs)
    results_4a[outcome] = res
    n_4a[outcome]       = n
    if res is not None:
        dfs_4a.append(fmt_res(res, n, dv=lead_col, label='4A'))

df_4a = pd.concat(dfs_4a, ignore_index=True)

print('Section 4A: Model A — Intensity Only')
for outcome in PRED_OUTCOMES:
    res = results_4a[outcome]
    if res is not None and 'intensity' in res.params.index:
        coef = float(res.params['intensity'])
        p    = float(res.pvalues['intensity'])
        se   = float(res.std_errors['intensity'])
        print(f'  lead_{outcome:<25s} N={n_4a[outcome]:,}  '
              f'intensity: {coef:+.4f} ({se:.4f}) {stars(p)}')
    else:
        print(f'  lead_{outcome:<25s} N={n_4a[outcome]:,}  result unavailable')

Section 4A: Model A — Intensity Only
  lead_firm_risk_w               N=6,519  intensity: +0.0291 (0.0230) 
  lead_beta_w                    N=6,500  intensity: -0.2737 (0.3141) 
  lead_ret_ann_w                 N=6,507  intensity: -0.3711 (0.3225) 
  lead_leverage_w                N=6,519  intensity: -0.0813 (0.0501) 
  lead_cash_flow_ratio_w         N=6,519  intensity: +0.0055 (0.0573) 
  lead_operating_cf_ratio_w      N=6,519  intensity: +0.0079 (0.0562) 


### 4B Intensity and one content dimension

In [41]:
results_4b = {}  # keyed by (outcome, content_var)
n_4b       = {}
dfs_4b     = []

for outcome in PRED_OUTCOMES:
    for cv in CONTENT_VARS:
        lead_col = f'lead_{outcome}'
        rhs = list(dict.fromkeys(['intensity', cv] + ALL_CONTROLS + [outcome]))
        res, n = run_panel_fe(panel, dv=lead_col, rhs_vars=rhs)
        results_4b[(outcome, cv)] = res
        n_4b[(outcome, cv)]       = n
        if res is not None:
            dfs_4b.append(fmt_res(res, n, dv=lead_col, label=f'4B_{cv}'))

df_4b = pd.concat(dfs_4b, ignore_index=True)

print(f'Section 4B: Model B ({len(PRED_OUTCOMES)} outcomes x {len(CONTENT_VARS)} content vars '
      f'= {len(PRED_OUTCOMES)*len(CONTENT_VARS)} regressions)')
print()
print('Example N per outcome (content var = specificity):')
for outcome in PRED_OUTCOMES:
    print(f'  lead_{outcome:<25s}  N = {n_4b[(outcome, "specificity")]:,}')

Section 4B: Model B (6 outcomes x 5 content vars = 30 regressions)

Example N per outcome (content var = specificity):
  lead_firm_risk_w                N = 5,660
  lead_beta_w                     N = 5,642
  lead_ret_ann_w                  N = 5,649
  lead_leverage_w                 N = 5,660
  lead_cash_flow_ratio_w          N = 5,660
  lead_operating_cf_ratio_w       N = 5,660


### 4C All six disclosure variables

In [42]:
results_4c = {}
n_4c       = {}
dfs_4c     = []

for outcome in PRED_OUTCOMES:
    lead_col = f'lead_{outcome}'
    rhs = list(dict.fromkeys(
        ['intensity'] + CONTENT_VARS + ALL_CONTROLS + [outcome]
    ))
    res, n = run_panel_fe(panel, dv=lead_col, rhs_vars=rhs)
    results_4c[outcome] = res
    n_4c[outcome]       = n
    if res is not None:
        dfs_4c.append(fmt_res(res, n, dv=lead_col, label='4C'))

df_4c = pd.concat(dfs_4c, ignore_index=True)

# ── CHECKPOINT 3 ──────────────────────────────────────────────────────────
print('=' * 65)
print('CHECKPOINT 3')
print('=' * 65)
print('Model A N per outcome:')
for o in PRED_OUTCOMES:
    print(f'  lead_{o:<25s}  N = {n_4a[o]:,}')
print()
print('Model B N (example: +specificity) per outcome:')
for o in PRED_OUTCOMES:
    print(f'  lead_{o:<25s}  N = {n_4b[(o,"specificity")]:,}')
print()
print('Model C N per outcome:')
for o in PRED_OUTCOMES:
    print(f'  lead_{o:<25s}  N = {n_4c[o]:,}')
print()
print('Lead outcome validation (non-null N < full panel N):')
for o in PRED_OUTCOMES:
    lead_n = panel[f'lead_{o}'].notna().sum()
    print(f'  lead_{o:<25s}: {lead_n:,} non-null (dropped {len(panel)-lead_n:,})')

CHECKPOINT 3
Model A N per outcome:
  lead_firm_risk_w                N = 6,519
  lead_beta_w                     N = 6,500
  lead_ret_ann_w                  N = 6,507
  lead_leverage_w                 N = 6,519
  lead_cash_flow_ratio_w          N = 6,519
  lead_operating_cf_ratio_w       N = 6,519

Model B N (example: +specificity) per outcome:
  lead_firm_risk_w                N = 5,660
  lead_beta_w                     N = 5,642
  lead_ret_ann_w                  N = 5,649
  lead_leverage_w                 N = 5,660
  lead_cash_flow_ratio_w          N = 5,660
  lead_operating_cf_ratio_w       N = 5,660

Model C N per outcome:
  lead_firm_risk_w                N = 5,427
  lead_beta_w                     N = 5,409
  lead_ret_ann_w                  N = 5,416
  lead_leverage_w                 N = 5,427
  lead_cash_flow_ratio_w          N = 5,427
  lead_operating_cf_ratio_w       N = 5,427

Lead outcome validation (non-null N < full panel N):
  lead_firm_risk_w              : 9,233 non-nu

In [43]:
df_4c.to_csv('4C_results.csv')

## 5 Persistence (Appendix Table VIII.3)

In [44]:
# Construct lags for NLP vars (only for consecutive years)
for var in NLP_VARS:
    panel[f'L_{var}'] = panel.groupby('cik')[var].shift(1)

consec = panel['is_consec']
panel_consec = panel.loc[consec].reset_index(drop=True)

# ── Raw AR(1): Pearson r over consecutive firm-year pairs ──────────────────
ar1_rows = []
for var in NLP_VARS:
    lag_col = f'L_{var}'
    sub = panel_consec[[var, lag_col]].dropna()
    if len(sub) > 10:
        r, p = pearsonr(sub[var], sub[lag_col])
        ar1_rows.append({'indicator': var, 'ar1_r': r, 'ar1_p': p, 'n': len(sub)})

df_ar1 = pd.DataFrame(ar1_rows)

# ── Within-firm persistence: PanelOLS with firm + year FE ─────────────────
persist_rows = []
persist_results = {}
persist_n       = {}

for var in NLP_VARS:
    lag_col = f'L_{var}'
    res, n  = run_panel_fe(panel_consec, dv=var, rhs_vars=[lag_col])
    persist_results[var] = res
    persist_n[var]       = n
    if res is not None and lag_col in res.params.index:
        p = float(res.pvalues[lag_col])
        persist_rows.append({
            'indicator': var,
            'gamma': float(res.params[lag_col]),
            'se':    float(res.std_errors[lag_col]),
            'tstat': float(res.tstats[lag_col]),
            'pval':  p,
            'stars': stars(p),
            'n':     n,
            'within_r2': float(res.rsquared),
        })

df_persist = pd.DataFrame(persist_rows)

# ── Annual cross-sectional means ───────────────────────────────────────────
annual_means = panel.groupby('fiscal_year')[NLP_VARS].mean().reset_index()

# ── CHECKPOINT 4 ──────────────────────────────────────────────────────────
print('=' * 65)
print('CHECKPOINT 4')
print('=' * 65)
print('Raw AR(1) Pearson r (consecutive firm-year pairs):')
print(df_ar1.to_string(index=False))
print()
print('Within-firm gamma from PanelOLS:')
print(df_persist[['indicator', 'gamma', 'se', 'stars', 'n', 'within_r2']].to_string(index=False))
print()
print('Annual cross-sectional means:')
print(annual_means.to_string(index=False))

CHECKPOINT 4
Raw AR(1) Pearson r (consecutive firm-year pairs):
   indicator    ar1_r    ar1_p    n
   intensity 0.882841 0.000000 9402
    geo_tone 0.826975 0.000000 7860
 specificity 0.769900 0.000000 7860
forward_look 0.768271 0.000000 7860
 uncertainty 0.803059 0.000000 7860
 novelty_geo 0.237944 0.000000 6381

Within-firm gamma from PanelOLS:
   indicator     gamma       se stars    n  within_r2
   intensity  0.430542 0.016806   *** 9402   0.186945
    geo_tone  0.354501 0.018758   *** 7860   0.132582
 specificity  0.347335 0.018900   *** 7860   0.123700
forward_look  0.321991 0.026094   *** 7860   0.103857
 uncertainty  0.309837 0.016424   *** 7860   0.100351
 novelty_geo -0.093428 0.017210   *** 6381   0.008926

Annual cross-sectional means:
 fiscal_year  intensity  geo_tone  specificity  forward_look  uncertainty  novelty_geo
        2018   0.034047  0.032249     0.007043      0.049066     1.101198          NaN
        2019   0.037593  0.032360     0.007886      0.046610     1.

In [45]:
df_ar1

,indicator,ar1_r,ar1_p,n
0,intensity,0.882841,0.000000,9402
1,geo_tone,0.826975,0.000000,7860
2,specificity,0.769900,0.000000,7860
3,forward_look,0.768271,0.000000,7860
4,uncertainty,0.803059,0.000000,7860
5,novelty_geo,0.237944,0.000000,6381


In [46]:
df_persist

,indicator,gamma,se,tstat,pval,stars,n,within_r2
0,intensity,0.430542,0.016806,25.618058,0.000000,***,9402,0.186945
1,geo_tone,0.354501,0.018758,18.898727,0.000000,***,7860,0.132582
2,specificity,0.347335,0.018900,18.377425,0.000000,***,7860,0.123700
3,forward_look,0.321991,0.026094,12.339789,0.000000,***,7860,0.103857
4,uncertainty,0.309837,0.016424,18.865111,0.000000,***,7860,0.100351
5,novelty_geo,-0.093428,0.017210,-5.428551,0.000000,***,6381,0.008926


## 6 Moderation of the predictive regressions

### 6A Foreign income share

In [47]:
# Controls for moderator regressions: ALL_CONTROLS + contemporaneous firm risk (lagged DV)
MOD_CONTROLS = ALL_CONTROLS + ['firm_risk_w']

# Spec 1: intensity × foreign_rev_share_w
res_6a1, n_6a1, std_6a1 = run_interaction_reg(
    panel, dv='lead_firm_risk_w',
    nlp_var='intensity', mod_var='foreign_rev_share_w',
    add_controls=MOD_CONTROLS
)

# Spec 2: novelty_geo × foreign_rev_share_w
res_6a2, n_6a2, std_6a2 = run_interaction_reg(
    panel, dv='lead_firm_risk_w',
    nlp_var='novelty_geo', mod_var='foreign_rev_share_w',
    add_controls=MOD_CONTROLS
)

dfs_6a = []
if res_6a1 is not None:
    dfs_6a.append(fmt_res(res_6a1, n_6a1, dv='lead_firm_risk_w', label='6A_Spec1'))
if res_6a2 is not None:
    dfs_6a.append(fmt_res(res_6a2, n_6a2, dv='lead_firm_risk_w', label='6A_Spec2'))
df_6a = pd.concat(dfs_6a, ignore_index=True)

print('Section 6A: H3b — Foreign Revenue Share Moderator')
print(f'  Spec 1 (intensity x FRS):   N = {n_6a1:,}')
print(f'  Spec 2 (novelty_geo x FRS): N = {n_6a2:,}')
for label, res, nlp in [('6A_Spec1', res_6a1, 'intensity'),
                         ('6A_Spec2', res_6a2, 'novelty_geo')]:
    int_key = f'{nlp}_x_foreign_rev_share_w'
    if res is not None and int_key in res.params.index:
        coef = float(res.params[int_key])
        p    = float(res.pvalues[int_key])
        se   = float(res.std_errors[int_key])
        print(f'  {label} B3: {coef:+.5f} ({se:.5f}) p={p:.3f} {stars(p)}')

Section 6A: H3b — Foreign Revenue Share Moderator
  Spec 1 (intensity x FRS):   N = 4,092
  Spec 2 (novelty_geo x FRS): N = 3,680
  6A_Spec1 B3: -0.00064 (0.00055) p=0.249 
  6A_Spec2 B3: +0.00043 (0.00048) p=0.379 


### 6B Analyst coverage

In [48]:
# Spec 1: specificity × log_analyst_cov
res_6b1, n_6b1, std_6b1 = run_interaction_reg(
    panel, dv='lead_firm_risk_w',
    nlp_var='specificity', mod_var='log_analyst_cov',
    add_controls=MOD_CONTROLS
)

# Spec 2: forward_look × log_analyst_cov
res_6b2, n_6b2, std_6b2 = run_interaction_reg(
    panel, dv='lead_firm_risk_w',
    nlp_var='forward_look', mod_var='log_analyst_cov',
    add_controls=MOD_CONTROLS
)

dfs_6b = []
if res_6b1 is not None:
    dfs_6b.append(fmt_res(res_6b1, n_6b1, dv='lead_firm_risk_w', label='6B_Spec1'))
if res_6b2 is not None:
    dfs_6b.append(fmt_res(res_6b2, n_6b2, dv='lead_firm_risk_w', label='6B_Spec2'))
df_6b = pd.concat(dfs_6b, ignore_index=True)

print('Section 6B: H3a — Analyst Coverage Moderator')
print(f'  Spec 1 (specificity x log_AC):   N = {n_6b1:,}')
print(f'  Spec 2 (forward_look x log_AC):  N = {n_6b2:,}')
for label, res, nlp in [('6B_Spec1', res_6b1, 'specificity'),
                         ('6B_Spec2', res_6b2, 'forward_look')]:
    int_key = f'{nlp}_x_log_analyst_cov'
    if res is not None and int_key in res.params.index:
        coef = float(res.params[int_key])
        p    = float(res.pvalues[int_key])
        se   = float(res.std_errors[int_key])
        print(f'  {label} B3: {coef:+.5f} ({se:.5f}) p={p:.3f} {stars(p)}')

# ── CHECKPOINT 5 ──────────────────────────────────────────────────────────
print()
print('=' * 65)
print('CHECKPOINT 5')
print('=' * 65)
print(f'H3b Spec1 (intensity x FRS):       N = {n_6a1:,}')
print(f'H3b Spec2 (novelty_geo x FRS):     N = {n_6a2:,}')
print(f'H3a Spec1 (specificity x log_AC):  N = {n_6b1:,}')
print(f'H3a Spec2 (forward_look x log_AC): N = {n_6b2:,}')
print()
print('Interaction B3 significance:')
for label, res, nlp, mod in [
    ('6A1 intensity x FRS',       res_6a1, 'intensity',    'foreign_rev_share_w'),
    ('6A2 novelty_geo x FRS',     res_6a2, 'novelty_geo',  'foreign_rev_share_w'),
    ('6B1 specificity x log_AC',  res_6b1, 'specificity',  'log_analyst_cov'),
    ('6B2 forward_look x log_AC', res_6b2, 'forward_look', 'log_analyst_cov'),
]:
    int_key = f'{nlp}_x_{mod}'
    if res is not None and int_key in res.params.index:
        coef = float(res.params[int_key])
        p    = float(res.pvalues[int_key])
        print(f'  {label:<30s}  B3 = {coef:+.5f}  p = {p:.3f}  {stars(p)}')
    else:
        print(f'  {label:<30s}  NOT FOUND IN PARAMS')
print()
print('Standardisation check (means ~0, stds ~1 within estimation sample):')
for label, std in [
    ('6A1 intensity/FRS', std_6a1), ('6A2 novelty/FRS', std_6a2),
    ('6B1 spec/log_AC',  std_6b1), ('6B2 fl/log_AC',   std_6b2),
]:
    if std is not None:
        means = std.loc['mean'].values
        stds  = std.loc['std'].values
        print(f'  {label:<22s}  means={[f"{v:.4f}" for v in means]}  '
              f'stds={[f"{v:.4f}" for v in stds]}')

Section 6B: H3a — Analyst Coverage Moderator
  Spec 1 (specificity x log_AC):   N = 5,660
  Spec 2 (forward_look x log_AC):  N = 5,660
  6B_Spec1 B3: +0.00068 (0.00052) p=0.188 
  6B_Spec2 B3: -0.00030 (0.00073) p=0.678 

CHECKPOINT 5
H3b Spec1 (intensity x FRS):       N = 4,092
H3b Spec2 (novelty_geo x FRS):     N = 3,680
H3a Spec1 (specificity x log_AC):  N = 5,660
H3a Spec2 (forward_look x log_AC): N = 5,660

Interaction B3 significance:
  6A1 intensity x FRS             B3 = -0.00064  p = 0.249  
  6A2 novelty_geo x FRS           B3 = +0.00043  p = 0.379  
  6B1 specificity x log_AC        B3 = +0.00068  p = 0.188  
  6B2 forward_look x log_AC       B3 = -0.00030  p = 0.678  

Standardisation check (means ~0, stds ~1 within estimation sample):
  6A1 intensity/FRS       means=['-0.0000', '0.0000']  stds=['1.0000', '1.0000']
  6A2 novelty/FRS         means=['-0.0000', '0.0000']  stds=['1.0000', '1.0000']
  6B1 spec/log_AC         means=['0.0000', '0.0000']  stds=['1.0000', '1.0000']


## 7 Novelty of the full Item 1A

In [49]:
ROB_NOV = ROBUSTNESS_NOVELTY  # 'novelty_full_item1a'

# ── Rob 3B: DV = novelty_full_item1a ──────────────────────────────────────
res_7_3b, n_7_3b = run_panel_fe(
    panel, dv=ROB_NOV, rhs_vars=['intensity'] + ALL_CONTROLS)
df_7_3b = fmt_res(res_7_3b, n_7_3b, dv=ROB_NOV, label='7_3B') if res_7_3b else pd.DataFrame()

# ── Rob 4C: horse-race with novelty_full_item1a replacing novelty_geo ──────
CONTENT_ROB = [c if c != 'novelty_geo' else ROB_NOV for c in CONTENT_VARS]
results_7_4c = {}
n_7_4c       = {}
dfs_7_4c     = []
for outcome in PRED_OUTCOMES:
    lead_col = f'lead_{outcome}'
    rhs = list(dict.fromkeys(
        ['intensity'] + CONTENT_ROB + ALL_CONTROLS + [outcome]
    ))
    res, n = run_panel_fe(panel, dv=lead_col, rhs_vars=rhs)
    results_7_4c[outcome] = res
    n_7_4c[outcome]       = n
    if res is not None:
        dfs_7_4c.append(fmt_res(res, n, dv=lead_col, label='7_4C'))

df_7_4c = pd.concat(dfs_7_4c, ignore_index=True) if dfs_7_4c else pd.DataFrame()

# ── Rob 6A: novelty_full_item1a × foreign_rev_share_w ─────────────────────
res_7_6a, n_7_6a, std_7_6a = run_interaction_reg(
    panel, dv='lead_firm_risk_w',
    nlp_var=ROB_NOV, mod_var='foreign_rev_share_w',
    add_controls=MOD_CONTROLS
)
df_7_6a = fmt_res(res_7_6a, n_7_6a, dv='lead_firm_risk_w', label='7_6A') if res_7_6a else pd.DataFrame()

# ── CHECKPOINT 6 ──────────────────────────────────────────────────────────
print('=' * 65)
print('CHECKPOINT 6')
print('=' * 65)
print(f'Rob 3B  (DV=novelty_full_item1a):           N = {n_7_3b:,}')
print('Rob 4C  (horse-race + novelty_full_item1a):')
for o in PRED_OUTCOMES:
    print(f'  lead_{o:<25s} N = {n_7_4c[o]:,}')
print(f'Rob 6A  (novelty_full x FRS):               N = {n_7_6a:,}')
print()

# Compare primary vs robustness novelty in 3B
print('Comparison: intensity coef on novelty_geo (primary) vs novelty_full_item1a (robustness):')
res_3b_geo  = results_3b.get('novelty_geo')
if res_3b_geo is not None and 'intensity' in res_3b_geo.params.index:
    c_geo  = float(res_3b_geo.params['intensity'])
    p_geo  = float(res_3b_geo.pvalues['intensity'])
    print(f'  novelty_geo:         coef = {c_geo:+.4f}  p = {p_geo:.3f}  {stars(p_geo)}')
if res_7_3b is not None and 'intensity' in res_7_3b.params.index:
    c_full = float(res_7_3b.params['intensity'])
    p_full = float(res_7_3b.pvalues['intensity'])
    print(f'  novelty_full_item1a: coef = {c_full:+.4f}  p = {p_full:.3f}  {stars(p_full)}')
    print()
    qual_similar = abs(c_full - c_geo) < 0.5 * abs(c_geo) and (p_full < 0.10) == (p_geo < 0.10)
    print(f'Qualitatively similar (same sign, both significant or not at 10%): {qual_similar}')

CHECKPOINT 6
Rob 3B  (DV=novelty_full_item1a):           N = 8,039
Rob 4C  (horse-race + novelty_full_item1a):
  lead_firm_risk_w               N = 5,660
  lead_beta_w                    N = 5,642
  lead_ret_ann_w                 N = 5,649
  lead_leverage_w                N = 5,660
  lead_cash_flow_ratio_w         N = 5,660
  lead_operating_cf_ratio_w      N = 5,660
Rob 6A  (novelty_full x FRS):               N = 4,092

Comparison: intensity coef on novelty_geo (primary) vs novelty_full_item1a (robustness):
  novelty_geo:         coef = +0.7761  p = 0.000  ***
  novelty_full_item1a: coef = +0.1610  p = 0.046  **

Qualitatively similar (same sign, both significant or not at 10%): False


## 8 Export tables

In [50]:
import shutil
from datetime import datetime

def safe_write_csv(df, path):
    """Write CSV, backing up existing file with timestamp before overwriting."""
    path = Path(path)
    if path.exists():
        ts     = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup = path.with_name(path.stem + f'_bak_{ts}' + path.suffix)
        shutil.copy2(path, backup)
        print(f'  [backed up] {backup.name}')
    df.to_csv(path, index=False)
    size_kb = path.stat().st_size / 1024
    print(f'  [written]   {path.name}  ({len(df):,} rows, {size_kb:.1f} KB)')

print('CHECKPOINT 7 — Exporting tables to', OUTPUT)
print()

# Section 3A: intensive-margin
safe_write_csv(df_3a, OUTPUT / 'tab_3b_intensive.csv')

# Section 3B: conditional content models
safe_write_csv(df_3b, OUTPUT / 'tab_3c_content.csv')

# Section 3C: H2 interaction
safe_write_csv(df_3c, OUTPUT / 'tab_3d_h2_fe.csv')

# Section 4: predictive (one file per outcome, all models A/B/C combined)
for outcome in PRED_OUTCOMES:
    lead_dv  = f'lead_{outcome}'
    rows_A   = df_4a[df_4a['dv'] == lead_dv]
    rows_B   = df_4b[df_4b['dv'] == lead_dv]
    rows_C   = df_4c[df_4c['dv'] == lead_dv]
    combined = pd.concat([rows_A, rows_B, rows_C], ignore_index=True)
    stub     = outcome.replace('_w', '')
    safe_write_csv(combined, OUTPUT / f'tab_5_pred_{stub}.csv')

# Section 5: persistence
safe_write_csv(df_ar1,      OUTPUT / 'tab_6_autocorr.csv')
safe_write_csv(df_persist,  OUTPUT / 'tab_6_persistence.csv')
safe_write_csv(annual_means, OUTPUT / 'tab_6_yearly_means.csv')

# Section 6A: H3b FRS moderator
safe_write_csv(df_6a[df_6a['label'] == '6A_Spec1'].copy(),
               OUTPUT / 'tab_7a_mod_intensity_frs.csv')
safe_write_csv(df_6a[df_6a['label'] == '6A_Spec2'].copy(),
               OUTPUT / 'tab_7a_mod_novelty_frs.csv')

# Section 6B: H3a AC moderator
safe_write_csv(df_6b[df_6b['label'] == '6B_Spec1'].copy(),
               OUTPUT / 'tab_7b_mod_specificity_ac.csv')
safe_write_csv(df_6b[df_6b['label'] == '6B_Spec2'].copy(),
               OUTPUT / 'tab_7b_mod_forward_look_ac.csv')

# Section 7: robustness — novelty_full_item1a
dfs_rob = [df for df in [df_7_3b, df_7_4c, df_7_6a] if not df.empty]
if dfs_rob:
    df_rob_all = pd.concat(dfs_rob, ignore_index=True)
    safe_write_csv(df_rob_all, OUTPUT / 'tab_rob_novelty_full.csv')

print()
print('CHECKPOINT 7 complete.')
print()

# Summary: list all files written
written = sorted(OUTPUT.glob('tab_*.csv'))
print(f'Files in {OUTPUT.name}/ matching tab_*.csv ({len(written)} total):')
for f in written:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<55s} {size_kb:6.1f} KB')

CHECKPOINT 7 — Exporting tables to c:\Users\Sören Schlißke\Dateien\Uni\MACFin\Thesis\Claude\output\tables

  [backed up] tab_3b_intensive_bak_20261114_104949.csv
  [written]   tab_3b_intensive.csv  (9 rows, 1.4 KB)
  [backed up] tab_3c_content_bak_20261114_104949.csv
  [written]   tab_3c_content.csv  (50 rows, 7.4 KB)
  [backed up] tab_3d_h2_fe_bak_20261114_104949.csv
  [written]   tab_3d_h2_fe.csv  (65 rows, 9.7 KB)
  [backed up] tab_5_pred_firm_risk_bak_20261114_104949.csv
  [written]   tab_5_pred_firm_risk.csv  (87 rows, 12.8 KB)
  [backed up] tab_5_pred_beta_bak_20261114_104949.csv
  [written]   tab_5_pred_beta.csv  (87 rows, 12.3 KB)
  [backed up] tab_5_pred_ret_ann_bak_20261114_104949.csv
  [written]   tab_5_pred_ret_ann.csv  (87 rows, 12.6 KB)
  [backed up] tab_5_pred_leverage_bak_20261114_104949.csv
  [written]   tab_5_pred_leverage.csv  (87 rows, 12.7 KB)
  [backed up] tab_5_pred_cash_flow_ratio_bak_20261114_104949.csv
  [written]   tab_5_pred_cash_flow_ratio.csv  (87 rows, 13

In [51]:
panel[NLP_VARS].corr().round(3)

,intensity,geo_tone,specificity,forward_look,uncertainty,novelty_geo
intensity,1.000000,-0.026000,0.153000,0.093000,0.185000,-0.040000
geo_tone,-0.026000,1.000000,0.136000,-0.081000,0.108000,0.040000
specificity,0.153000,0.136000,1.000000,-0.046000,-0.144000,0.126000
forward_look,0.093000,-0.081000,-0.046000,1.000000,0.107000,-0.007000
uncertainty,0.185000,0.108000,-0.144000,0.107000,1.000000,-0.095000
novelty_geo,-0.040000,0.040000,0.126000,-0.007000,-0.095000,1.000000


## 9 Thesis tables: standardised effects, novelty comparison, filing-window tests

In [52]:
def bh(p):
    """Benjamini-Hochberg q-values."""
    p = np.asarray(p, float); n = len(p); order = np.argsort(p)
    q = p[order] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[order] = np.minimum(q, 1)
    return out

def effect_rows(results, match, label, interaction):
    """Coefficient, 95% CI, effect in within-firm SDs of the DV, and the minimum effect
    detectable with 80% power at 5% (2.8 x SE), on the same scale.
    Main effects: per within-firm SD of x. Interactions (z-scored interactants): per 1 SD."""
    rows = []
    for dv, res in results.items():
        if res is None:
            continue
        keys = [k for k in res.params.index if match(str(k))]
        if not keys:
            continue
        k = keys[0]
        b, se = float(res.params[k]), float(res.std_errors[k])
        y = res.model.dependent.dataframe.iloc[:, 0]
        sd_y = (y - y.groupby(level=0).transform('mean')).std()
        if interaction:
            sx = 1.0
        else:
            x = res.model.exog.dataframe[k]
            sx = (x - x.groupby(level=0).transform('mean')).std()
        s = sx / sd_y
        rows.append({'table': label, 'dv': dv, 'term': k, 'coef': b, 'se': se,
                     'ci_lo': b - 1.96 * se, 'ci_hi': b + 1.96 * se, 'p': float(res.pvalues[k]),
                     'n': int(res.nobs), 'std_effect': b * s, 'std_ci_lo': (b - 1.96 * se) * s,
                     'std_ci_hi': (b + 1.96 * se) * s, 'mde80_std': 2.8 * se * s})
    return rows

df_eff = pd.DataFrame(
    effect_rows(results_3b,     lambda k: k == 'intensity',          '3B intensity',       False)
  + effect_rows(results_3c,     lambda k: 'x_prop_cost' in k,        '3C x PropCost',      True)
  + effect_rows(results_3c_lit, lambda k: 'x_high_litigation' in k,  '3C x litigation',    True))
print(df_eff[['table', 'dv', 'coef', 'p', 'std_effect', 'std_ci_lo', 'std_ci_hi', 'mde80_std', 'n']]
      .round(4).to_string(index=False))
df_eff.to_csv(OUTPUT / 'tab_effect_sizes_h2.csv', index=False)

          table           dv      coef        p  std_effect  std_ci_lo  std_ci_hi  mde80_std    n
   3B intensity  specificity  0.006700 0.296400    0.025500  -0.022300   0.073300   0.068300 7107
   3B intensity forward_look  0.224900 0.000300    0.086500   0.040200   0.132700   0.066100 7107
   3B intensity  uncertainty  1.358900 0.000000    0.126800   0.076800   0.176800   0.071500 7107
   3B intensity     geo_tone -0.034000 0.003700   -0.080900  -0.135500  -0.026300   0.078000 7107
   3B intensity  novelty_geo  0.776100 0.000000    0.122100   0.078500   0.165800   0.062400 6817
  3C x PropCost  specificity -0.000000 0.789800   -0.010200  -0.084900   0.064600   0.106800 6671
  3C x PropCost forward_look -0.001800 0.382000   -0.037900  -0.122800   0.047000   0.121300 6671
  3C x PropCost  uncertainty -0.002100 0.773100   -0.010600  -0.082900   0.061600   0.103200 6671
  3C x PropCost     geo_tone  0.000300 0.408100    0.034300  -0.046900   0.115500   0.116000 6671
  3C x PropCost  nov

In [53]:
pl_path = ROOT / 'data/features/novelty_placebo.csv'
EXTRA = ['novelty_nongeo', 'novelty_placebo']
pl = pd.read_csv(pl_path)[['cik', 'fiscal_year'] + EXTRA]
panel = panel.drop(columns=EXTRA, errors='ignore').merge(pl, on=['cik', 'fiscal_year'], how='left')
for c in EXTRA:
    lo, hi = panel[c].quantile([0.01, 0.99])
    panel[c] = panel[c].clip(lo, hi)

NOV = ['novelty_geo', 'novelty_full_item1a'] + EXTRA
common = panel.dropna(subset=NOV + ['intensity'] + ALL_CONTROLS).copy()
print(f'Common sample: N = {len(common):,}')
print(common[NOV].describe().loc[['mean', 'std']].round(3).to_string())

res_nov = {dv: run_panel_fe(common, dv=dv, rhs_vars=['intensity'] + ALL_CONTROLS)[0] for dv in NOV}
df_nov = pd.DataFrame(effect_rows(res_nov, lambda k: k == 'intensity', 'novelty, common sample', False))
print(df_nov[['dv', 'coef', 'se', 'p', 'std_effect', 'std_ci_lo', 'std_ci_hi', 'n']].round(4).to_string(index=False))

def stacked_diff(df, dv_a, dv_b):
    """Tests whether intensity's standardised effect differs between two DVs: stacks both
    (each standardised), DV-specific firm FE, year FE and control slopes; clusters by firm."""
    parts = []
    for dv, g in [(dv_a, 1), (dv_b, 0)]:
        d = df[['cik', 'fiscal_year', 'intensity'] + ALL_CONTROLS].copy()
        d['y'], d['g'] = (df[dv] - df[dv].mean()) / df[dv].std(), g
        parts.append(d)
    L = pd.concat(parts, ignore_index=True)
    L['x'] = (L['intensity'] - L['intensity'].mean()) / L['intensity'].std()
    L['x_g'] = L['x'] * L['g']
    ctrl = []
    for c in ALL_CONTROLS:
        L[c + '_g'] = L[c] * L['g']
        ctrl += [c, c + '_g']
    L['ent'] = L['cik'].astype(str) + '_' + L['g'].astype(str)
    L['t'] = L['fiscal_year'] * 10 + L['g']
    L = L.set_index(['ent', 't'])
    clusters = pd.DataFrame({'firm': L['cik'].astype('int64')}, index=L.index)
    r = PanelOLS(L['y'], L[['x', 'x_g'] + ctrl], entity_effects=True, time_effects=True,
                 drop_absorbed=True).fit(cov_type='clustered', clusters=clusters)
    return float(r.params['x_g']), float(r.std_errors['x_g']), float(r.pvalues['x_g'])

diff_rows = []
for b in ['novelty_full_item1a', 'novelty_nongeo', 'novelty_placebo']:
    d, se, p = stacked_diff(common, 'novelty_geo', b)
    diff_rows.append({'contrast': f'novelty_geo - {b}', 'diff_std_coef': d, 'se': se, 'p': p})
    print(f'novelty_geo - {b}: difference in standardised coefficient = {d:+.3f} (SE {se:.3f}, p = {p:.3f})')
pd.concat([df_nov, pd.DataFrame(diff_rows)], ignore_index=True).to_csv(
    OUTPUT / 'tab_rob_novelty_common_placebo.csv', index=False)

Common sample: N = 6,817
      novelty_geo  novelty_full_item1a  novelty_nongeo  novelty_placebo
mean     0.167000             0.107000        0.106000         0.429000
std      0.145000             0.086000        0.085000         0.161000
                 dv      coef       se        p  std_effect  std_ci_lo  std_ci_hi    n
        novelty_geo  0.776100 0.141600 0.000000    0.122100   0.078500   0.165800 6817
novelty_full_item1a  0.194900 0.087600 0.026100    0.052400   0.006200   0.098600 6817
     novelty_nongeo  0.217200 0.084900 0.010600    0.059600   0.013900   0.105300 6817
    novelty_placebo -0.417700 0.115300 0.000300   -0.070400  -0.108400  -0.032300 6817
novelty_geo - novelty_full_item1a: difference in standardised coefficient = +0.127 (SE 0.052, p = 0.015)
novelty_geo - novelty_nongeo: difference in standardised coefficient = +0.115 (SE 0.051, p = 0.026)
novelty_geo - novelty_placebo: difference in standardised coefficient = +0.328 (SE 0.038, p = 0.000)


In [54]:
strict = panel.copy()
prev_ok   = strict.groupby('cik')['fiscal_year'].shift(1) == strict['fiscal_year'] - 1
prev_thin = strict.groupby('cik')['thin_strict'].shift(1).astype('boolean').fillna(False).astype(bool)
cur       = strict['thin_strict'].astype(bool)
strict.loc[cur, ['geo_tone', 'specificity', 'forward_look', 'uncertainty']] = np.nan
strict.loc[cur | (prev_ok & prev_thin), 'novelty_geo'] = np.nan
print('Content obs (specificity), main vs strict:',
      panel['specificity'].notna().sum(), strict['specificity'].notna().sum())

rows = []
def keep(sample, test, dv, res, n, terms):
    if res is None:
        return
    for k in res.params.index:
        if any(t == k or t in k for t in terms):
            rows.append({'sample': sample, 'test': test, 'dv': dv, 'var': k,
                         'coef': float(res.params[k]), 'p': float(res.pvalues[k]), 'n': n})

for name, d_ in [('main 3/100', panel), ('strict 5/200', strict)]:
    for cv in CONTENT_VARS:
        res, n = run_panel_fe(d_, dv=cv, rhs_vars=['intensity'] + ALL_CONTROLS)
        keep(name, '3B', cv, res, n, ['intensity'])
        res, n, _ = run_interaction_reg(d_, dv=cv, nlp_var='intensity', mod_var='prop_cost',
                                        add_controls=ALL_CONTROLS + ['rd_missing'])
        keep(name, '3C PropCost', cv, res, n, ['x_prop_cost'])
        res, n, _ = run_interaction_reg(d_, dv=cv, nlp_var='intensity', mod_var='high_litigation',
                                        add_controls=ALL_CONTROLS)
        keep(name, '3C litigation', cv, res, n, ['x_high_litigation'])
    for o in PRED_OUTCOMES:
        rhs = list(dict.fromkeys(['intensity'] + CONTENT_VARS + ALL_CONTROLS + [o]))
        res, n = run_panel_fe(d_, dv=f'lead_{o}', rhs_vars=rhs)
        keep(name, '4C', f'lead_{o}', res, n, ['intensity'] + CONTENT_VARS)

df_strict = pd.DataFrame(rows)
wide = df_strict.pivot_table(index=['test', 'dv', 'var'], columns='sample', values=['coef', 'p', 'n'])
print(wide.round(4).to_string())
df_strict.to_csv(OUTPUT / 'tab_rob_thin_strict.csv', index=False)

Content obs (specificity), main vs strict: 9489 8080
                                                                          coef                        n                       p             
sample                                                              main 3/100 strict 5/200  main 3/100 strict 5/200 main 3/100 strict 5/200
test          dv                        var                                                                                                 
3B            forward_look              intensity                     0.224900     0.238000 7107.000000  6181.000000   0.000300     0.000100
              geo_tone                  intensity                    -0.034000    -0.033900 7107.000000  6181.000000   0.003700     0.004000
              novelty_geo               intensity                     0.776100     0.658000 6817.000000  5678.000000   0.000000     0.000000
              specificity               intensity                     0.006700     0.000300 7107.0000

In [55]:
# 9.3b  Geo-Tone without the mitigation component (low agreement on the mitigation component in the validation)
neg = pd.read_parquet(ROOT / 'data/features/nlp_features.parquet',
                      columns=['cik', 'fiscal_year', 'n_negative_hits_geo'])
panel = panel.drop(columns=['n_negative_hits_geo', 'geo_tone_neg'], errors='ignore').merge(
    neg, on=['cik', 'fiscal_year'], how='left')
panel['geo_tone_neg'] = (panel['n_negative_hits_geo'] / panel['geo_token_count']).where(panel['geo_tone'].notna())
lo, hi = panel['geo_tone_neg'].quantile([0.01, 0.99])
panel['geo_tone_neg'] = panel['geo_tone_neg'].clip(lo, hi)
print('corr(geo_tone, geo_tone_neg) =', round(panel[['geo_tone', 'geo_tone_neg']].corr().iloc[0, 1], 3))

tone_rows = []
for dv in ['geo_tone', 'geo_tone_neg']:
    res, n = run_panel_fe(panel, dv=dv, rhs_vars=['intensity'] + ALL_CONTROLS)
    tone_rows += effect_rows({dv: res}, lambda k: k == 'intensity', '3B tone variants', False)
    res, n, _ = run_interaction_reg(panel, dv=dv, nlp_var='intensity', mod_var='prop_cost',
                                    add_controls=ALL_CONTROLS + ['rd_missing'])
    tone_rows += effect_rows({dv: res}, lambda k: 'x_prop_cost' in k, '3C tone variants', True)
df_tone = pd.DataFrame(tone_rows)
print(df_tone[['table', 'dv', 'coef', 'p', 'std_effect', 'std_ci_lo', 'std_ci_hi', 'n']].round(4).to_string(index=False))
df_tone.to_csv(OUTPUT / 'tab_rob_tone_negative_only.csv', index=False)

corr(geo_tone, geo_tone_neg) = 0.954
           table           dv      coef        p  std_effect  std_ci_lo  std_ci_hi    n
3B tone variants     geo_tone -0.034000 0.003700   -0.080900  -0.135500  -0.026300 7107
3C tone variants     geo_tone  0.000300 0.408100    0.034300  -0.046900   0.115500 6671
3B tone variants geo_tone_neg -0.039000 0.000700   -0.097400  -0.153500  -0.041300 7107
3C tone variants geo_tone_neg  0.000100 0.817000    0.009800  -0.073400   0.093000 6671


In [56]:
c4 = df_4c[df_4c['variable'].isin(['intensity'] + CONTENT_VARS)].copy()
c4['q_bh'] = bh(c4['pval'])
print(c4[['dv', 'variable', 'coef', 'pval', 'q_bh', 'n']].sort_values('pval').round(4).head(10).to_string(index=False))
print(f"\nNLP coefficients at p < .10: {(c4['pval'] < .10).sum()} of {len(c4)} "
      f"(expected by chance: {0.10 * len(c4):.1f}); at q < .10: {(c4['q_bh'] < .10).sum()}")
c4.to_csv(OUTPUT / 'tab_5_modelC_bh.csv', index=False)

                    dv     variable      coef     pval     q_bh    n
        lead_ret_ann_w    intensity -0.596000 0.087800 0.784700 5416
           lead_beta_w  specificity  1.546800 0.115200 0.784700 5409
        lead_ret_ann_w forward_look  0.193300 0.150400 0.784700 5416
       lead_leverage_w forward_look -0.024100 0.166800 0.784700 5427
lead_cash_flow_ratio_w forward_look -0.026400 0.175100 0.784700 5427
      lead_firm_risk_w    intensity  0.035900 0.175400 0.784700 5427
           lead_beta_w  uncertainty  0.033300 0.190300 0.784700 5409
lead_cash_flow_ratio_w  novelty_geo  0.010600 0.199200 0.784700 5427
       lead_leverage_w  uncertainty -0.005700 0.215500 0.784700 5427
       lead_leverage_w    intensity -0.069400 0.218000 0.784700 5427

NLP coefficients at p < .10: 1 of 36 (expected by chance: 3.6); at q < .10: 0


In [57]:
dly = pd.read_parquet(ROOT / 'data/intermediate/crsp_daily_returns_around_filing.parquet',
                      columns=['cik', 'fiscal_year', 'trading_day_rel', 'ret', 'mkt_ret', 'vol'])
dly = dly.dropna(subset=['trading_day_rel']).sort_values(['cik', 'fiscal_year', 'trading_day_rel'])
for c in ['ret', 'mkt_ret', 'vol']:
    dly[c] = pd.to_numeric(dly[c], errors='coerce').to_numpy(dtype='float64', na_value=np.nan)

def event_measures(g):
    t = g['trading_day_rel'].to_numpy(float); r = g['ret'].to_numpy(float)
    m = g['mkt_ret'].to_numpy(float);         v = g['vol'].to_numpy(float)
    est = (t >= -250) & (t <= -11) & np.isfinite(r) & np.isfinite(m)
    out = dict.fromkeys(['sigma_pre', 'car01', 'car03', 'car03_mkt', 'abvol', 'abturn03'], np.nan)
    out['n_est'] = int(est.sum())      # every event returns the same keys -> apply() yields a DataFrame
    if est.sum() < 120:
        return pd.Series(out, dtype=float)
    beta, alpha = np.polyfit(m[est], r[est], 1)
    ar = r - (alpha + beta * m)
    sigma_pre = np.std(ar[est], ddof=2)
    win = lambda a, b: (t >= a) & (t <= b) & np.isfinite(ar)
    w01, w03, wpost = win(0, 1), win(0, 3), win(6, 28)
    out['sigma_pre'] = sigma_pre
    out['car01']     = ar[w01].sum() if w01.sum() == 2 else np.nan
    out['car03']     = ar[w03].sum() if w03.sum() == 4 else np.nan
    out['car03_mkt'] = (r - m)[w03].sum() if w03.sum() == 4 else np.nan
    out['abvol']     = (np.log(np.std(ar[wpost], ddof=1) / sigma_pre)
                        if wpost.sum() >= 15 and sigma_pre > 0 else np.nan)
    vpre = v[(t >= -250) & (t <= -11) & np.isfinite(v)]
    v03  = v[(t >= 0) & (t <= 3) & np.isfinite(v)]
    out['abturn03'] = (np.log(v03.mean() / vpre.mean())
                       if len(v03) == 4 and len(vpre) >= 120 and vpre.mean() > 0 and v03.mean() > 0 else np.nan)
    return pd.Series(out, dtype=float)

ev = (dly.groupby(['cik', 'fiscal_year'])[['trading_day_rel', 'ret', 'mkt_ret', 'vol']]
         .apply(event_measures).reset_index())
ev['abs_car03'] = ev['car03'].abs()
for c in ['car01', 'car03', 'car03_mkt', 'abs_car03', 'abvol', 'abturn03', 'sigma_pre']:
    lo, hi = ev[c].quantile([0.01, 0.99])
    ev[c + '_w'] = ev[c].clip(lo, hi)
ev['cik'] = ev['cik'].astype('int64'); ev['fiscal_year'] = ev['fiscal_year'].astype('int64')

rq2 = panel.assign(cik=panel['cik'].astype('int64')).merge(ev, on=['cik', 'fiscal_year'], how='left')
rq2 = rq2.sort_values(['cik', 'fiscal_year']).reset_index(drop=True)

# Changes (consecutive years only)
prev = rq2.groupby('cik')['fiscal_year'].shift(1) == rq2['fiscal_year'] - 1
for c in ['intensity', 'log_item1a_tokens']:
    rq2['d_' + c] = rq2.groupby('cik')[c].diff().where(prev)

# Geopolitical climate at filing: C&I GPR in the month before the filing month
gpr = pd.read_excel(ROOT / 'data/validation/external/data_gpr_export.xls', sheet_name='Sheet1')[['month', 'GPR']]
gpr['month'] = pd.to_datetime(gpr['month']).dt.to_period('M')
rq2['gpr_pre'] = (pd.to_datetime(rq2['filing_date']).dt.to_period('M') - 1).map(gpr.set_index('month')['GPR'])

# Earnings announcement within [-1, +5] calendar days of the filing date
rdq = pd.read_parquet(ROOT / 'data/intermediate/rdq_q4.parquet')
rdq['gvkey'] = rdq['gvkey'].astype(str).str.zfill(6)
rq2['gvkey_s'] = rq2['gvkey'].astype(str).str.split('.').str[0].str.zfill(6)
rq2['datadate'] = pd.to_datetime(rq2['datadate'])
rq2 = rq2.merge(rdq.rename(columns={'gvkey': 'gvkey_s'}), on=['gvkey_s', 'datadate'], how='left')
gap = (rq2['rdq'] - pd.to_datetime(rq2['filing_date'])).dt.days
rq2['ea_in_window'] = gap.between(-1, 5)

RQ2_Y    = ['abs_car03_w', 'car03_w', 'abvol_w', 'abturn03_w']
RQ2_CTRL = ['L_log_mktcap', 'L_book_to_market_w', 'L_log_item1a_tokens', 'sigma_pre_w', 'L_log_analyst_cov']
print('missing controls:', [c for c in RQ2_CTRL if c not in rq2.columns])
print('\nEvent coverage by fiscal year (share with CAR[0,3]):')
print(rq2.groupby('fiscal_year')['car03'].apply(lambda x: x.notna().mean()).round(3).to_string())
print(f"\nEarnings release within [-1,+5] days of filing: {rq2['ea_in_window'].mean():.1%}")
print(rq2[['car03', 'abs_car03', 'abvol', 'abturn03', 'gpr_pre']].describe().round(4).to_string())

missing controls: []

Event coverage by fiscal year (share with CAR[0,3]):
fiscal_year
2018   0.983000
2019   0.983000
2020   0.983000
2021   0.991000
2022   0.994000
2023   0.993000
2024   0.995000
2025   0.217000

Earnings release within [-1,+5] days of filing: 36.1%
            car03   abs_car03       abvol    abturn03      gpr_pre
count 9714.000000 9714.000000 9695.000000 9713.000000 10962.000000
mean     0.001200    0.041800   -0.019200    0.204100   122.611300
std      0.065700    0.050700    0.577200    0.462100    35.292300
min     -0.640300    0.000000   -1.975000   -1.740100    58.420800
25%     -0.025100    0.011100   -0.369600   -0.113400    92.016500
50%      0.000200    0.025500   -0.130500    0.165900   113.233200
75%      0.025700    0.052600    0.151800    0.474400   143.162400
max      0.629200    0.640300    3.596100    2.587300   318.954900


In [58]:
def run_yearfe(df, dv, rhs_vars, entity_col='cik', time_col='fiscal_year', ind_col='ff48_industry'):
    cols = list(dict.fromkeys([entity_col, time_col, ind_col, dv] + rhs_vars))
    sub = df[cols].dropna()
    if len(sub) < 20:
        return None, len(sub)
    sub = sub.set_index([entity_col, time_col])
    res = PanelOLS(sub[dv], sub[rhs_vars], time_effects=True,
                   other_effects=sub[[ind_col]].astype(int), drop_absorbed=True
                   ).fit(cov_type='clustered', cluster_entity=True)
    return res, len(sub)

rq2_clean = rq2[~rq2['ea_in_window'].fillna(False).astype(bool)]
out = []
def add(res, n, label, y, terms):
    if res is not None:
        d = fmt_res(res, n, dv=y, label=label)
        out.append(d[d['variable'].isin(terms)])

for y in RQ2_Y:
    res, n = run_panel_fe(rq2, dv=y, rhs_vars=['intensity'] + RQ2_CTRL)
    add(res, n, 'R1 intensity', y, ['intensity'])
    res, n = run_panel_fe(rq2, dv=y, rhs_vars=['intensity'] + CONTENT_VARS + RQ2_CTRL)
    add(res, n, 'R2 horse race', y, ['intensity'] + CONTENT_VARS)
    res, n = run_yearfe(rq2, y, ['d_intensity', 'novelty_geo', 'd_log_item1a_tokens'] + RQ2_CTRL)
    add(res, n, 'R3 changes', y, ['d_intensity', 'novelty_geo'])
    res, n, _ = run_interaction_reg(rq2, dv=y, nlp_var='intensity', mod_var='gpr_pre', add_controls=RQ2_CTRL)
    add(res, n, 'R4 x GPR', y, ['intensity_std', 'gpr_pre_std', 'intensity_x_gpr_pre'])
    res, n = run_panel_fe(rq2_clean, dv=y, rhs_vars=['intensity'] + CONTENT_VARS + RQ2_CTRL)
    add(res, n, 'R2 excl. earnings overlap', y, ['intensity'] + CONTENT_VARS)

df_rq2 = pd.concat(out, ignore_index=True)
fam = df_rq2['label'] == 'R2 horse race'
df_rq2.loc[fam, 'q_bh'] = bh(df_rq2.loc[fam, 'pval'])
print(df_rq2[['label', 'dv', 'variable', 'coef', 'se', 'pval', 'stars', 'q_bh', 'n']]
      .round(4).to_string(index=False))
df_rq2.to_csv(OUTPUT / 'tab_rq2_filing.csv', index=False)

                    label          dv            variable      coef       se     pval stars     q_bh    n
             R1 intensity abs_car03_w           intensity  0.014000 0.032300 0.664700            NaN 8212
            R2 horse race abs_car03_w           intensity  0.033600 0.035900 0.348700       0.901200 6792
            R2 horse race abs_car03_w         specificity  0.005200 0.123300 0.966600       0.966600 6792
            R2 horse race abs_car03_w        forward_look  0.008000 0.013000 0.538300       0.901200 6792
            R2 horse race abs_car03_w         uncertainty -0.002500 0.003000 0.413300       0.901200 6792
            R2 horse race abs_car03_w            geo_tone  0.097600 0.076000 0.199300       0.901200 6792
            R2 horse race abs_car03_w         novelty_geo  0.000900 0.005400 0.871900       0.966600 6792
               R3 changes abs_car03_w         d_intensity  0.016500 0.028600 0.562800            NaN 6792
               R3 changes abs_car03_w         

In [59]:
# 9.7  Robustness for the RQ2 volatility result
# (a) filing-month FE replace year FE: absorbs market-wide volatility regimes (e.g. March 2020),
#     so identification comes from firms filing in the same calendar month;
# (b) two-way clustering by firm and filing month; (c) without earnings-overlap filings.
rq2['fym_id'] = pd.to_datetime(rq2['filing_date']).dt.to_period('M').astype(str).astype('category').cat.codes
rq2_clean = rq2[~rq2['ea_in_window'].fillna(False).astype(bool)]
rows97 = []
for sample_name, d_ in [('all', rq2), ('excl. earnings overlap', rq2_clean)]:
    for y in ['abvol_w', 'abs_car03_w', 'abturn03_w']:
        for spec, rhs in [('intensity', ['intensity']), ('horse race', ['intensity'] + CONTENT_VARS)]:
            sub = (d_[['cik', 'fym_id', y] + rhs + RQ2_CTRL].dropna()
                   .drop_duplicates(['cik', 'fym_id'], keep='last').set_index(['cik', 'fym_id']))
            r = PanelOLS(sub[y], sub[rhs + RQ2_CTRL], entity_effects=True, time_effects=True,
                         drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True, cluster_time=True)
            rows97.append({'sample': sample_name, 'dv': y, 'spec': spec,
                           'coef_intensity': float(r.params['intensity']),
                           'se': float(r.std_errors['intensity']), 'p': float(r.pvalues['intensity']),
                           'n': int(r.nobs)})
df97 = pd.DataFrame(rows97)
print(df97.round(4).to_string(index=False))

# Economic magnitude: a one-within-firm-SD change in intensity, in percent of post-filing volatility
x = rq2[['cik', 'intensity']].dropna()
sd_within = (x['intensity'] - x.groupby('cik')['intensity'].transform('mean')).std()
b = df97.query("sample == 'all' and dv == 'abvol_w' and spec == 'intensity'")['coef_intensity'].iloc[0]
print(f'\nwithin-firm SD of intensity = {sd_within:.4f}; effect on ln(vol ratio) = {b * sd_within:+.4f} '
      f'(= {100 * (np.exp(b * sd_within) - 1):+.1f}% post-filing idiosyncratic volatility)')
df97.to_csv(OUTPUT / 'tab_rq2_robust_fym.csv', index=False)

                sample          dv       spec  coef_intensity       se        p    n
                   all     abvol_w  intensity        0.192400 0.308800 0.533100 8193
                   all     abvol_w horse race        0.160100 0.297700 0.590800 6776
                   all abs_car03_w  intensity        0.007900 0.035000 0.821700 8212
                   all abs_car03_w horse race        0.028600 0.041600 0.492300 6792
                   all  abturn03_w  intensity        0.096800 0.272000 0.722000 8211
                   all  abturn03_w horse race       -0.059100 0.289600 0.838200 6792
excl. earnings overlap     abvol_w  intensity        0.175500 0.331400 0.596500 5226
excl. earnings overlap     abvol_w horse race        0.163900 0.336300 0.626000 4400
excl. earnings overlap abs_car03_w  intensity        0.011600 0.026300 0.659600 5240
excl. earnings overlap abs_car03_w horse race        0.026500 0.036900 0.471900 4411
excl. earnings overlap  abturn03_w  intensity       -0.068800 0.3

In [60]:
NLP6 = ['intensity'] + CONTENT_VARS


def within_sd(s):
    """SD after removing firm means (firm = level 0 of the panel index)."""
    return float((s - s.groupby(level=0).transform('mean')).std())


def std_row(res, term, label, dv, sx=None, sx_col=None):
    """One coefficient, standardised as in section 3.6.4.

    sx      fixed regressor scale, e.g. 1.0 for z-scored moderators and the GPR
            index (per 1 SD of that variable)
    sx_col  take the within-firm SD from another regressor column; for an
            interaction of z-scores this is the z-scored disclosure variable,
            so the effect is per within-firm SD of disclosure and per 1 SD of
            the moderator
    """
    y = res.model.dependent.dataframe.iloc[:, 0]
    X = res.model.exog.dataframe
    if sx is None:
        sx = within_sd(X[sx_col or term])
    s = sx / within_sd(y)
    b, se = float(res.params[term]), float(res.std_errors[term])
    return {'table': label, 'dv': dv, 'term': term, 'coef_raw': b, 'se_raw': se,
            'p': float(res.pvalues[term]), 'n': int(res.nobs),
            'std_coef': b * s, 'std_se': se * s,
            'std_ci_lo': (b - 1.96 * se) * s, 'std_ci_hi': (b + 1.96 * se) * s,
            'mde80_std': 2.8 * se * s}


def wide(df, label, paren='se', rows=None, cols=None):
    """Paste-ready layout: 'coef*** (SE)' or 'coef*** (p)', three decimals."""
    d = df[df['table'] == label].copy()
    inner = d['std_se'] if paren == 'se' else d['p']
    d['cell'] = [f'{c:.3f}{stars(p)} ({v:.3f})' for c, p, v in zip(d['std_coef'], d['p'], inner)]
    out = d.pivot(index='term', columns='dv', values='cell')
    return out.reindex(index=rows or list(dict.fromkeys(d['term'])),
                       columns=cols or list(dict.fromkeys(d['dv'])))


# Table 13 and Appendix Table VIII.4: filing-window tests, standardised
def run_fym(d_, y, rhs):
    """Filing-month FE model of section 9.7: firm + filing-month FE,
    SEs clustered by firm and filing month."""
    sub = (d_[['cik', 'fym_id', y] + rhs + RQ2_CTRL].dropna()
           .drop_duplicates(['cik', 'fym_id'], keep='last').set_index(['cik', 'fym_id']))
    return PanelOLS(sub[y], sub[rhs + RQ2_CTRL], entity_effects=True, time_effects=True,
                    drop_absorbed=True).fit(cov_type='clustered', cluster_entity=True,
                                            cluster_time=True)


In [61]:
T12_Y = ['abvol_w', 'abs_car03_w', 'abturn03_w']
rows = []
for y in T12_Y:
    rows.append(std_row(run_fym(rq2, y, ['intensity']), 'intensity', 'FM intensity alone', y))
    r = run_fym(rq2, y, NLP6)
    rows += [std_row(r, v, 'FM horse race', y) for v in NLP6]
    rows.append(std_row(run_fym(rq2_clean, y, ['intensity']), 'intensity', 'FM without earnings overlap', y))
    r, _ = run_panel_fe(rq2, dv=y, rhs_vars=['intensity'] + RQ2_CTRL)   # firm + fiscal-year FE, firm clusters
    rows.append(std_row(r, 'intensity', 'FY intensity alone (comparison)', y))
t12 = pd.DataFrame(rows)
hr = t12['table'] == 'FM horse race'
t12.loc[hr, 'q_bh'] = bh(t12.loc[hr, 'p'])   # FDR family: 6 disclosure variables x 3 outcomes

print('Table 12, intensity rows (std. coefficient, SE in parentheses):')
t12_int = t12[t12['term'] == 'intensity']
display(pd.concat({lab: wide(t12_int, lab, cols=T12_Y)
                   for lab in ['FM intensity alone', 'FM horse race',
                               'FM without earnings overlap', 'FY intensity alone (comparison)']}))
print('\nFiling-month horse race, all six coefficients (for the H3 text or an appendix table):')
display(wide(t12, 'FM horse race', rows=NLP6, cols=T12_Y))
print('q-values (BH), filing-month horse race:')
display(t12.loc[hr].pivot(index='term', columns='dv', values='q_bh').reindex(NLP6)[T12_Y].round(3))
t12.to_csv(OUTPUT / 'tab12_filing_window_std.csv', index=False)

Table 12, intensity rows (std. coefficient, SE in parentheses):


,dv,abvol_w,abs_car03_w,abturn03_w
,term,,,
FM intensity alone,intensity,0.006 (0.010),0.004 (0.017),0.005 (0.014)
FM horse race,intensity,0.006 (0.011),0.015 (0.021),-0.003 (0.015)
FM without earnings overlap,intensity,0.006 (0.011),0.008 (0.019),-0.004 (0.017)
FY intensity alone (comparison),intensity,0.031*** (0.010),0.007 (0.016),0.018 (0.014)



Filing-month horse race, all six coefficients (for the H3 text or an appendix table):


dv,abvol_w,abs_car03_w,abturn03_w
term,,,
intensity,0.006 (0.011),0.015 (0.021),-0.003 (0.015)
specificity,0.002 (0.007),0.003 (0.014),0.003 (0.012)
forward_look,-0.002 (0.007),0.006 (0.015),-0.001 (0.013)
uncertainty,-0.001 (0.010),-0.009 (0.016),-0.013 (0.012)
geo_tone,0.001 (0.008),0.019 (0.014),-0.026* (0.014)
novelty_geo,0.001 (0.009),0.005 (0.020),0.016 (0.014)


q-values (BH), filing-month horse race:


dv,abvol_w,abs_car03_w,abturn03_w
term,,,
intensity,0.966000,0.966000,0.966000
specificity,0.966000,0.966000,0.966000
forward_look,0.966000,0.966000,0.966000
uncertainty,0.966000,0.966000,0.966000
geo_tone,0.966000,0.966000,0.966000
novelty_geo,0.966000,0.966000,0.966000


In [62]:
# Table 14 (horse race) and Appendix Table VIII.1 (intensity alone, one dimension at a time), standardised
rows = []
for o in PRED_OUTCOMES:
    rows += [std_row(results_4c[o], v, 'T13 horse race', o) for v in NLP6]
    rows.append(std_row(results_4a[o], 'intensity', 'VIII.1 intensity alone', o))
    rows += [std_row(results_4b[(o, cv)], cv, 'VIII.1 one content dimension', o) for cv in CONTENT_VARS]
t13 = pd.DataFrame(rows)

print('Table 13 (std. coefficient, p in parentheses; q-values unchanged):')
display(wide(t13, 'T13 horse race', paren='p', rows=NLP6, cols=PRED_OUTCOMES))
print('\nTable VIII.1, column 1 (Model A) and columns 2-6 (Model B), p in parentheses:')
display(pd.concat([wide(t13, 'VIII.1 intensity alone', paren='p', cols=PRED_OUTCOMES),
                   wide(t13, 'VIII.1 one content dimension', paren='p', rows=CONTENT_VARS,
                        cols=PRED_OUTCOMES)]))
t13.to_csv(OUTPUT / 'tab13_viii1_predictive_std.csv', index=False)

Table 13 (std. coefficient, p in parentheses; q-values unchanged):


dv,firm_risk_w,beta_w,ret_ann_w,leverage_w,cash_flow_ratio_w,operating_cf_ratio_w
term,,,,,,
intensity,0.028 (0.175),-0.014 (0.437),-0.030* (0.088),-0.026 (0.218),-0.014 (0.547),-0.008 (0.776)
specificity,-0.007 (0.688),0.024 (0.115),-0.000 (0.999),-0.010 (0.596),-0.013 (0.462),0.004 (0.838)
forward_look,0.002 (0.903),-0.007 (0.717),0.025 (0.150),-0.023 (0.167),-0.026 (0.175),-0.020 (0.265)
uncertainty,-0.016 (0.371),0.019 (0.190),0.007 (0.689),-0.022 (0.216),0.009 (0.595),-0.008 (0.699)
geo_tone,-0.013 (0.334),0.004 (0.805),-0.000 (0.980),-0.007 (0.670),0.002 (0.883),0.002 (0.895)
novelty_geo,-0.015 (0.381),0.012 (0.431),0.014 (0.360),0.000 (0.979),0.027 (0.199),0.009 (0.627)



Table VIII.1, column 1 (Model A) and columns 2-6 (Model B), p in parentheses:


dv,firm_risk_w,beta_w,ret_ann_w,leverage_w,cash_flow_ratio_w,operating_cf_ratio_w
term,,,,,,
intensity,0.022 (0.205),-0.015 (0.384),-0.018 (0.250),-0.029 (0.104),0.002 (0.923),0.003 (0.888)
specificity,-0.010 (0.513),0.023 (0.116),-0.003 (0.849),-0.000 (0.999),-0.013 (0.450),-0.003 (0.904)
forward_look,0.002 (0.919),-0.008 (0.658),0.025 (0.121),-0.022 (0.173),-0.031* (0.082),-0.021 (0.233)
uncertainty,-0.017 (0.303),0.015 (0.271),0.009 (0.541),-0.030* (0.086),0.008 (0.599),-0.009 (0.651)
geo_tone,-0.021 (0.121),0.007 (0.619),0.001 (0.915),-0.010 (0.527),0.008 (0.587),0.007 (0.695)
novelty_geo,-0.015 (0.403),0.012 (0.438),0.014 (0.366),0.001 (0.941),0.026 (0.218),0.010 (0.612)


In [63]:
# Appendix Tables VIII.2 (changes, GPR interaction) and VII.1 (moderators), standardised
rows = []
for y in RQ2_Y:
    r, _ = run_yearfe(rq2, y, ['d_intensity', 'novelty_geo', 'd_log_item1a_tokens'] + RQ2_CTRL)
    rows += [std_row(r, v, 'VIII.2 changes', y) for v in ['d_intensity', 'novelty_geo']]
    r, _, _ = run_interaction_reg(rq2, dv=y, nlp_var='intensity', mod_var='gpr_pre',
                                  add_controls=RQ2_CTRL)
    rows += [std_row(r, 'intensity_std', 'VIII.2 x GPR', y),                    # per within-firm SD of intensity
             std_row(r, 'gpr_pre_std', 'VIII.2 x GPR', y, sx=1.0),             # per SD of the GPR index
             std_row(r, 'intensity_x_gpr_pre', 'VIII.2 x GPR', y, sx_col='intensity_std')]

for lab, r, nlp, mod in [('VII.1 intensity x FRS', res_6a1, 'intensity', 'foreign_rev_share_w'),
                         ('VII.1 novelty x FRS', res_6a2, 'novelty_geo', 'foreign_rev_share_w'),
                         ('VII.1 specificity x AC', res_6b1, 'specificity', 'log_analyst_cov'),
                         ('VII.1 forward-looking x AC', res_6b2, 'forward_look', 'log_analyst_cov')]:
    rows += [std_row(r, f'{nlp}_std', lab, 'lead_firm_risk_w'),                # per within-firm SD of disclosure
             std_row(r, f'{mod}_std', lab, 'lead_firm_risk_w', sx=1.0),        # per SD of the moderator
             std_row(r, f'{nlp}_x_{mod}', lab, 'lead_firm_risk_w', sx_col=f'{nlp}_std')]
t_app = pd.DataFrame(rows)

print('Table VIII.2 (std. coefficient, p in parentheses):')
display(pd.concat([wide(t_app, 'VIII.2 changes', paren='p', cols=RQ2_Y),
                   wide(t_app, 'VIII.2 x GPR', paren='p', cols=RQ2_Y)]))
print('\nTable VII.1 (std. coefficient, p in parentheses):')
display(t_app[t_app['table'].str.startswith('VII.1')]
        .assign(cell=lambda d: [f'{c:.3f}{stars(p)} ({p:.3f})' for c, p in zip(d['std_coef'], d['p'])])
        [['table', 'term', 'cell', 'n']])
t_app.to_csv(OUTPUT / 'tab_viii2_vii1_std.csv', index=False)

Table VIII.2 (std. coefficient, p in parentheses):


dv,abs_car03_w,car03_w,abvol_w,abturn03_w
term,,,,
d_intensity,0.009 (0.563),-0.005 (0.755),0.001 (0.895),0.016 (0.274)
novelty_geo,-0.004 (0.754),-0.016 (0.194),-0.001 (0.925),-0.008 (0.534)
intensity_std,0.007 (0.675),-0.013 (0.431),0.028*** (0.003),0.016 (0.267)
gpr_pre_std,-0.005 (0.804),0.041** (0.044),0.155*** (0.000),0.090*** (0.000)
intensity_x_gpr_pre,0.003 (0.653),0.010 (0.166),-0.000 (0.983),0.005 (0.404)



Table VII.1 (std. coefficient, p in parentheses):


,table,term,cell,n
20,VII.1 intensity x FRS,intensity_std,0.034 (0.128),4092
21,VII.1 intensity x FRS,foreign_rev_share_w_std,-0.015 (0.597),4092
22,VII.1 intensity x FRS,intensity_x_foreign_rev_share_w,-0.011 (0.249),4092
23,VII.1 novelty x FRS,novelty_geo_std,-0.010 (0.634),3680
24,VII.1 novelty x FRS,foreign_rev_share_w_std,-0.029 (0.371),3680
25,VII.1 novelty x FRS,novelty_geo_x_foreign_rev_share_w,0.015 (0.379),3680
26,VII.1 specificity x AC,specificity_std,-0.009 (0.577),5660
27,VII.1 specificity x AC,log_analyst_cov_std,-0.121* (0.088),5660
28,VII.1 specificity x AC,specificity_x_log_analyst_cov,0.017 (0.188),5660
29,VII.1 forward-looking x AC,forward_look_std,0.002 (0.886),5660


In [64]:
# Table 12: interactions per within-firm SD of intensity
#       and per total SD of intensity
rows = []
for lab, results, mod in [('x PropCost', results_3c, 'prop_cost'),
                          ('x litigation', results_3c_lit, 'high_litigation')]:
    for cv, r in results.items():
        if r is None:
            continue
        cur = std_row(r, f'intensity_x_{mod}', lab, cv, sx=1.0)
        new = std_row(r, f'intensity_x_{mod}', lab, cv, sx_col='intensity_std')
        rows.append({**new, 'std_coef_current': cur['std_coef'], 'mde80_current': cur['mde80_std']})
t11 = pd.DataFrame(rows)
print('Table 11: current scale vs per within-firm SD of intensity (p unchanged)')
display(t11[['table', 'dv', 'std_coef_current', 'mde80_current',
             'std_coef', 'std_ci_lo', 'std_ci_hi', 'mde80_std', 'p', 'n']].round(3))
t11.to_csv(OUTPUT / 'tab11_interactions_within.csv', index=False)

Table 11: current scale vs per within-firm SD of intensity (p unchanged)


,table,dv,std_coef_current,mde80_current,std_coef,std_ci_lo,std_ci_hi,mde80_std,p,n
0,x PropCost,specificity,-0.010000,0.107000,-0.004000,-0.037000,0.028000,0.047000,0.790000,6671
1,x PropCost,forward_look,-0.038000,0.121000,-0.017000,-0.054000,0.021000,0.053000,0.382000,6671
2,x PropCost,uncertainty,-0.011000,0.103000,-0.005000,-0.036000,0.027000,0.045000,0.773000,6671
3,x PropCost,geo_tone,0.034000,0.116000,0.015000,-0.021000,0.051000,0.051000,0.408000,6671
4,x PropCost,novelty_geo,-0.009000,0.095000,-0.004000,-0.033000,0.026000,0.042000,0.800000,6412
5,x litigation,specificity,0.051000,0.101000,0.023000,-0.009000,0.054000,0.044000,0.155000,7075
6,x litigation,forward_look,-0.010000,0.104000,-0.004000,-0.037000,0.028000,0.046000,0.789000,7075
7,x litigation,uncertainty,-0.044000,0.123000,-0.020000,-0.058000,0.019000,0.054000,0.315000,7075
8,x litigation,geo_tone,-0.019000,0.131000,-0.008000,-0.049000,0.032000,0.058000,0.683000,7075
9,x litigation,novelty_geo,-0.016000,0.098000,-0.007000,-0.037000,0.023000,0.043000,0.651000,6787


In [65]:
# Table 11: stacked difference test on the within-firm scale
def stacked_diff_within(df, dv_a, dv_b):
    """As stacked_diff in cell 49, but outcome and intensity are scaled by their
    WITHIN-FIRM SDs on the common sample, so the estimate equals the difference
    between the two Table 10 effects."""
    def wsd(c):
        return (df[c] - df.groupby('cik')[c].transform('mean')).std()
    parts = []
    for dv, g in [(dv_a, 1), (dv_b, 0)]:
        d = df[['cik', 'fiscal_year', 'intensity'] + ALL_CONTROLS].copy()
        d['y'], d['g'] = df[dv] / wsd(dv), g
        parts.append(d)
    L = pd.concat(parts, ignore_index=True)
    L['x'] = L['intensity'] / wsd('intensity')
    L['x_g'] = L['x'] * L['g']
    ctrl = []
    for c in ALL_CONTROLS:
        L[c + '_g'] = L[c] * L['g']
        ctrl += [c, c + '_g']
    L['ent'] = L['cik'].astype(str) + '_' + L['g'].astype(str)
    L['t'] = L['fiscal_year'] * 10 + L['g']
    L = L.set_index(['ent', 't'])
    clusters = pd.DataFrame({'firm': L['cik'].astype('int64')}, index=L.index)
    r = PanelOLS(L['y'], L[['x', 'x_g'] + ctrl], entity_effects=True, time_effects=True,
                 drop_absorbed=True).fit(cov_type='clustered', clusters=clusters)
    return float(r.params['x_g']), float(r.std_errors['x_g']), float(r.pvalues['x_g'])


eff = df_nov.set_index('dv')['std_effect']
rows = []
for b in ['novelty_full_item1a', 'novelty_nongeo', 'novelty_placebo']:
    d, se, p = stacked_diff_within(common, 'novelty_geo', b)
    rows.append({'contrast': f'novelty_geo - {b}', 'difference': d, 'se': se, 'p': p,
                 'check: effect_geo - effect_b': eff['novelty_geo'] - eff[b]})
t10 = pd.DataFrame(rows)
print('Table 10, difference column on the within-firm scale (should equal the check column):')
display(t10.round(3))
t10.to_csv(OUTPUT / 'tab10_stacked_within.csv', index=False)

Table 10, difference column on the within-firm scale (should equal the check column):


,contrast,difference,se,p,check: effect_geo - effect_b
0,novelty_geo - novelty_full_item1a,0.070000,0.029000,0.017000,0.070000
1,novelty_geo - novelty_nongeo,0.063000,0.029000,0.030000,0.063000
2,novelty_geo - novelty_placebo,0.193000,0.022000,0.000000,0.193000


In [66]:
# Tables 3, 6 and 7 and the section 4.1 figures on winsorised indicators
#       (run after the filing-window preparation cell, which winsorises novelty_nongeo and novelty_placebo;
#        NLP_VARS and novelty_full_item1a are winsorised at 1/99 in Section 2)
T3_VARS = NLP_VARS + ['novelty_full_item1a', 'novelty_nongeo']
d = panel[['cik', 'fiscal_year'] + T3_VARS].copy()

t3 = pd.DataFrame({'N': d[T3_VARS].count(), 'Mean': d[T3_VARS].mean(), 'SD': d[T3_VARS].std(),
                   'Median': d[T3_VARS].median(), 'P5': d[T3_VARS].quantile(0.05),
                   'P95': d[T3_VARS].quantile(0.95)})
print('Table 3 (indicators winsorised at the 1st and 99th percentiles):')
display(t3.round(3))

t6 = d.groupby('fiscal_year').agg(N=('intensity', 'size'), **{v: (v, 'mean') for v in NLP_VARS})
t6.index = t6.index.astype(str)
t6.loc['Total'] = [len(d)] + [d[v].mean() for v in NLP_VARS]
t6['N'] = t6['N'].astype(int)
print('\nTable 6 (annual means, winsorised; N = all filings):')
display(t6.round(3))

m = d.groupby('fiscal_year')['intensity']
mean_y, med_y = m.mean(), m.median()
print(f'\nSection 4.1: mean intensity {mean_y[2018]:.4f} (2018) -> {mean_y[2025]:.4f} (2025), '
      f'{mean_y[2025] / mean_y[2018] - 1:+.1%}; median {med_y[2018]:.4f} -> {med_y[2025]:.4f}')
w = d.pivot(index='cik', columns='fiscal_year', values='intensity')
for a, b in [(2021, 2022), (2018, 2024), (2018, 2025)]:
    both = w[[a, b]].dropna()
    print(f'Share of firms with higher intensity in {b} than in {a}: '
          f'{(both[b] > both[a]).mean():.3f} (n = {len(both):,})')

print('\nTable 7 (pairwise correlations, winsorised):')
display(d[NLP_VARS].corr().round(3))
dm = d[NLP_VARS] - d.groupby('cik')[NLP_VARS].transform('mean')
print('Within-firm correlations (firm means removed), for the sentence below Table 7:')
display(dm.corr().round(3))
t3.to_csv(OUTPUT / 'tab3_descriptives_w.csv')
t6.to_csv(OUTPUT / 'tab6_annual_means_w.csv')

Table 3 (indicators winsorised at the 1st and 99th percentiles):


,N,Mean,SD,Median,P5,P95
intensity,10962,0.047000,0.040000,0.036000,0.002000,0.128000
geo_tone,9489,0.033000,0.016000,0.032000,0.009000,0.063000
specificity,9489,0.008000,0.009000,0.006000,0.000000,0.026000
forward_look,9489,0.057000,0.086000,0.000000,0.000000,0.250000
uncertainty,9489,1.119000,0.403000,1.083000,0.500000,1.850000
novelty_geo,7860,0.167000,0.146000,0.125000,0.007000,0.477000
novelty_full_item1a,9402,0.113000,0.097000,0.090000,0.024000,0.322000
novelty_nongeo,7860,0.109000,0.090000,0.088000,0.023000,0.295000



Table 6 (annual means, winsorised; N = all filings):


,N,intensity,geo_tone,specificity,forward_look,uncertainty,novelty_geo
fiscal_year,,,,,,,
2018,1183,0.034000,0.032000,0.007000,0.049000,1.101000,NaN
2019,1272,0.038000,0.032000,0.008000,0.047000,1.101000,0.161000
2020,1320,0.038000,0.032000,0.007000,0.053000,1.111000,0.182000
2021,1399,0.039000,0.033000,0.007000,0.053000,1.124000,0.139000
2022,1424,0.050000,0.035000,0.010000,0.061000,1.099000,0.199000
2023,1437,0.052000,0.034000,0.010000,0.060000,1.116000,0.156000
2024,1465,0.057000,0.034000,0.009000,0.064000,1.132000,0.150000
2025,1462,0.063000,0.034000,0.007000,0.067000,1.151000,0.183000
Total,10962,0.047000,0.033000,0.008000,0.057000,1.119000,0.167000



Section 4.1: mean intensity 0.0340 (2018) -> 0.0632 (2025), +85.6%; median 0.0242 -> 0.0550
Share of firms with higher intensity in 2022 than in 2021: 0.748 (n = 1,394)
Share of firms with higher intensity in 2024 than in 2018: 0.764 (n = 1,176)
Share of firms with higher intensity in 2025 than in 2018: 0.811 (n = 1,162)

Table 7 (pairwise correlations, winsorised):


,intensity,geo_tone,specificity,forward_look,uncertainty,novelty_geo
intensity,1.000000,-0.026000,0.153000,0.093000,0.185000,-0.040000
geo_tone,-0.026000,1.000000,0.136000,-0.081000,0.108000,0.040000
specificity,0.153000,0.136000,1.000000,-0.046000,-0.144000,0.126000
forward_look,0.093000,-0.081000,-0.046000,1.000000,0.107000,-0.007000
uncertainty,0.185000,0.108000,-0.144000,0.107000,1.000000,-0.095000
novelty_geo,-0.040000,0.040000,0.126000,-0.007000,-0.095000,1.000000


Within-firm correlations (firm means removed), for the sentence below Table 7:


,intensity,geo_tone,specificity,forward_look,uncertainty,novelty_geo
intensity,1.000000,-0.013000,0.083000,0.104000,0.116000,0.081000
geo_tone,-0.013000,1.000000,0.108000,-0.015000,0.119000,-0.010000
specificity,0.083000,0.108000,1.000000,-0.004000,-0.152000,0.038000
forward_look,0.104000,-0.015000,-0.004000,1.000000,0.063000,0.004000
uncertainty,0.116000,0.119000,-0.152000,0.063000,1.000000,-0.033000
novelty_geo,0.081000,-0.010000,0.038000,0.004000,-0.033000,1.000000
